# Shape-memory alloy analysis

This notebook records feature selection, model fitting, candidate screening, and
transformation-strain calculations. The model trials remain in their original
order; later cells reuse dataframes, selected features, and fitted models.

## Before running

- Fill in the three spreadsheet URLs and local file paths in the configuration
  cell below. Supply your own service-account JSON file with access to the sheets.
- Install the analysis dependencies listed in `README.md` and provide the input
  CSV files required by the sections you intend to run.
- The thermodynamic job section generates scripts and submits Slurm jobs. Use it
  only after configuring the templates for your computing environment.

Saved outputs and execution counts have been cleared. Markdown results describe
historical runs and are not a new execution of this cleaned notebook.

In [ ]:
# Shared analysis imports; run this cell before the configuration and analysis cells.
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap

pd.options.mode.copy_on_write = True


## Data-source configuration

Set each placeholder used by your analysis sections here. The worksheet names
remain at their call sites because they identify the expected table layouts.

In [ ]:
from pathlib import Path

# Replace these placeholders with your own data sources before loading a sheet.
SERVICE_ACCOUNT_FILE = "<SERVICE_ACCOUNT_JSON_PATH>"
LITERATURE_SPREADSHEET_URL = "<LITERATURE_SPREADSHEET_URL>"
ITERATION_1_SPREADSHEET_URL = "<ITERATION_1_SPREADSHEET_URL>"
ITERATION_2_SPREADSHEET_URL = "<ITERATION_2_SPREADSHEET_URL>"

# These are public OAuth permission identifiers, not private document links.
GOOGLE_API_SCOPES = [
    "https://spreadsheets.google.com/feeds",
    "https://www.googleapis.com/auth/drive",
]

# Replace the paths used by the optional candidate-screening and job sections.
COMPOSITION_SPACE_CSV = "<COMPOSITION_SPACE_CSV_PATH>"
GENERATED_COMPOSITION_SPACE_CSV = "<GENERATED_COMPOSITION_SPACE_CSV_PATH>"
PREVIOUS_TC_INPUT_CSV = "<PREVIOUS_TC_INPUT_CSV_PATH>"
PREVIOUS_MS_PREDICTIONS_CSV = "<PREVIOUS_MS_PREDICTIONS_CSV_PATH>"
TC_RESULTS_DIRECTORY = Path("<TC_RESULTS_DIRECTORY>")

# Create one authenticated client on the first read; no network access at setup.
_sheet_client = None


def get_dataframe(spreadsheet_url, worksheet_name):
    """Read worksheet values and suffix repeated column names with their counts.

    Numeric conversion stays in each analysis cell because column selections and
    target preparation differ between experiments. Re-run this setup cell after
    changing credentials to reset the cached spreadsheet client.
    """
    global _sheet_client

    if str(spreadsheet_url).startswith("<"):
        raise ValueError(
            "Replace the spreadsheet URL placeholder in the configuration cell."
        )
    if str(SERVICE_ACCOUNT_FILE).startswith("<"):
        raise ValueError("Set SERVICE_ACCOUNT_FILE to your service-account JSON path.")

    if _sheet_client is None:
        credentials = ServiceAccountCredentials.from_json_keyfile_name(
            SERVICE_ACCOUNT_FILE, GOOGLE_API_SCOPES
        )
        _sheet_client = gspread.authorize(credentials)

    worksheet = _sheet_client.open_by_url(spreadsheet_url).worksheet(worksheet_name)
    values = worksheet.get_all_values()
    if not values:
        return pd.DataFrame()

    # Preserve the original occurrence-based suffixes for duplicate headers.
    header = values[0].copy()
    counts = {}
    for index, name in enumerate(header):
        count = counts.get(name, 0)
        counts[name] = count + 1
        if count > 0:
            header[index] = f"{name}_{count}"
    return pd.DataFrame(values[1:], columns=header)


## Transformation-temperature feature selection

Load the literature and experimental iterations, check the transformation
temperature ordering, and generate composition descriptors. The initial split
trains on the literature and first iteration and evaluates on the second.

In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
outliers = df_lit[(df_lit["Total"] > 100.5) | (df_lit["Total"] < 99.5)]
# Save outliers to CSV
outliers.to_csv("drops_output.csv", index=False)
print(len(outliers))
# Drop outliers from df_lit
df_lit = df_lit[~df_lit.index.isin(outliers.index)]

df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)
df_lit


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :7].columns.to_list()].sum(axis=1)
outliers = df_iter1[(df_iter1["Total"] > 100.5) | (df_iter1["Total"] < 99.5)]
# Save outliers to CSV
outliers.to_csv("drops_output2.csv", index=False)
print(len(outliers))
df_iter1 = df_iter1[~df_iter1.index.isin(outliers.index)]

df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)
df_iter1


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_2_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]


df_iter2 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_iter2.iloc[:, :7] = df_iter2.iloc[:, :7] * 100
df_iter2.loc[:, "Total"] = df_iter2[df_iter2.iloc[:, :7].columns.to_list()].sum(axis=1)
outliers = df_iter2[(df_iter2["Total"] > 100.5) | (df_iter2["Total"] < 99.5)]
# Save outliers to CSV
outliers.to_csv("drops_output3.csv", index=False)
print(len(outliers))
df_iter2 = df_iter2[~df_iter2.index.isin(outliers.index)]

df_iter2 = df_iter2.drop("Total", axis=1)
df_iter2["Hysteresis"] = df_iter2["SME_Af"] - df_iter2["SME_Ms"]
df_iter2.drop_duplicates(inplace=True)
df_iter2.reset_index(inplace=True, drop=True)
df_iter2


In [ ]:
# Initial data and feature generation
analyzer = FeatureGenerator(df_lit.iloc[:, :40])
analyzer.generate_composition_formula()
main_elements = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
X_features = analyzer.generate_features_all(main_elements=main_elements)
# Concatenate the features and additional columns at once
X_lit = pd.concat([X_features, df_lit.iloc[:, 40:45]], axis=1)
X_lit


In [ ]:
# Ensure all required columns are present in df_iter1
required_columns = [
    "Ag",
    "Al",
    "Au",
    "B",
    "Bi",
    "Ce",
    "Co",
    "Cr",
    "Cu",
    "Dy",
    "Er",
    "Fe",
    "Ga",
    "Gd",
    "Hf",
    "In",
    "La",
    "Mn",
    "Mo",
    "Nb",
    "Nd",
    "Ni",
    "Pb",
    "Pd",
    "Pr",
    "Pt",
    "Re",
    "Rh",
    "Sb",
    "Sc",
    "Si",
    "Sn",
    "Ta",
    "Te",
    "Ti",
    "Tl",
    "V",
    "W",
    "Y",
    "Zr",
]
for column in required_columns:
    if column not in df_iter1.columns:
        df_iter1[column] = 0

# Select the required columns
df_iter1_elnames = df_iter1[required_columns]

# Initial data and feature generation
analyzer = FeatureGenerator(df_iter1_elnames)
analyzer.generate_composition_formula()
main_elements = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
X_features = analyzer.generate_features_all(main_elements=main_elements)
X_iter1 = pd.concat([X_features, df_iter1.iloc[:, 7:12]], axis=1)
X_iter1


In [ ]:
# Ensure all required columns are present in df_iter2
required_columns = [
    "Ag",
    "Al",
    "Au",
    "B",
    "Bi",
    "Ce",
    "Co",
    "Cr",
    "Cu",
    "Dy",
    "Er",
    "Fe",
    "Ga",
    "Gd",
    "Hf",
    "In",
    "La",
    "Mn",
    "Mo",
    "Nb",
    "Nd",
    "Ni",
    "Pb",
    "Pd",
    "Pr",
    "Pt",
    "Re",
    "Rh",
    "Sb",
    "Sc",
    "Si",
    "Sn",
    "Ta",
    "Te",
    "Ti",
    "Tl",
    "V",
    "W",
    "Y",
    "Zr",
]
for column in required_columns:
    if column not in df_iter2.columns:
        df_iter2[column] = 0

# Select the required columns
df_iter2_elnames = df_iter2[required_columns]

# Initial data and feature generation
analyzer = FeatureGenerator(df_iter2_elnames)
analyzer.generate_composition_formula()
main_elements = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
X_features = analyzer.generate_features_all(main_elements=main_elements)
X_iter2 = pd.concat([X_features, df_iter2.iloc[:, 7:12]], axis=1)
X_iter2


In [ ]:
X_train = pd.concat([X_lit, X_iter1], axis=0).reset_index(drop=True)
y_train = pd.concat([df_lit.iloc[:, -5:], df_iter1.iloc[:, 12:17]], axis=0).reset_index(
    drop=True
)
X_train


In [ ]:
y_train


In [ ]:
X_test = X_iter2.copy()
X_test


In [ ]:
y_test = df_iter2.iloc[:, 12:17].copy()
y_test


### Remove highly correlated features

In [ ]:
%%time
# Keep one feature from each group whose absolute correlation exceeds 0.9.
import pandas as pd

# Combine X_train and X_test
X_combined = pd.concat([X_train, X_test], axis=0)

# Calculate absolute correlation matrix
correlation_matrix = X_combined.corr().abs()

# Select the upper triangle of the correlation matrix
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find features to drop (correlation > 0.9)
features_to_drop = [
    column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)
]

# Drop identified features from X_train and X_test
X_train_filtered = X_train.drop(columns=features_to_drop)
X_test_filtered = X_test.drop(columns=features_to_drop)

# Save the remaining features in X_train_filtered to P_C_features.csv
X_train_filtered.columns.to_series().to_csv("P_C_features.csv", index=False)

# Return the filtered datasets
X_train_filtered, X_test_filtered


In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["SME_Ms"], pred[:, 0]))
r2 = r2_score(y_test["SME_Ms"], pred[:, 0])
mae = mean_absolute_error(y_test["SME_Ms"], pred[:, 0])
print("================Ms=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Ms")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Ms"], pred[:, 0])
xmin = min(min(y_test["SME_Ms"]), min(pred[:, 0])) - 10
xmax = max(max(y_test["SME_Ms"]), max(pred[:, 0])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Mf"], pred[:, 1]))
r2 = r2_score(y_test["SME_Mf"], pred[:, 1])
mae = mean_absolute_error(y_test["SME_Mf"], pred[:, 1])
print("================Mf=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Mf")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Mf"], pred[:, 1])
xmin = min(min(y_test["SME_Mf"]), min(pred[:, 1])) - 10
xmax = max(max(y_test["SME_Mf"]), max(pred[:, 1])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Af"], pred[:, 2]))
r2 = r2_score(y_test["SME_Af"], pred[:, 2])
mae = mean_absolute_error(y_test["SME_Af"], pred[:, 2])
print("================Af=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Af")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Af"], pred[:, 2])
xmin = min(min(y_test["SME_Af"]), min(pred[:, 2])) - 10
xmax = max(max(y_test["SME_Af"]), max(pred[:, 2])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_As"], pred[:, 3]))
r2 = r2_score(y_test["SME_As"], pred[:, 3])
mae = mean_absolute_error(y_test["SME_As"], pred[:, 3])
print("================As=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("As")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_As"], pred[:, 3])
xmin = min(min(y_test["SME_As"]), min(pred[:, 3])) - 10
xmax = max(max(y_test["SME_As"]), max(pred[:, 3])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred[:, 4]))
r2 = r2_score(y_test["Hysteresis"], pred[:, 4])
mae = mean_absolute_error(y_test["Hysteresis"], pred[:, 4])
print("=============Hysteresis==============")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred[:, 4])
xmin = min(min(y_test["Hysteresis"]), min(pred[:, 4])) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred[:, 4])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train_filtered.shape[1] - 2074


### Compare a larger feature subset

In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=13,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["SME_Ms"], pred[:, 0]))
r2 = r2_score(y_test["SME_Ms"], pred[:, 0])
mae = mean_absolute_error(y_test["SME_Ms"], pred[:, 0])
print("================Ms=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Ms")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Ms"], pred[:, 0])
xmin = min(min(y_test["SME_Ms"]), min(pred[:, 0])) - 10
xmax = max(max(y_test["SME_Ms"]), max(pred[:, 0])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Mf"], pred[:, 1]))
r2 = r2_score(y_test["SME_Mf"], pred[:, 1])
mae = mean_absolute_error(y_test["SME_Mf"], pred[:, 1])
print("================Mf=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Mf")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Mf"], pred[:, 1])
xmin = min(min(y_test["SME_Mf"]), min(pred[:, 1])) - 10
xmax = max(max(y_test["SME_Mf"]), max(pred[:, 1])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Af"], pred[:, 2]))
r2 = r2_score(y_test["SME_Af"], pred[:, 2])
mae = mean_absolute_error(y_test["SME_Af"], pred[:, 2])
print("================Af=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Af")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Af"], pred[:, 2])
xmin = min(min(y_test["SME_Af"]), min(pred[:, 2])) - 10
xmax = max(max(y_test["SME_Af"]), max(pred[:, 2])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_As"], pred[:, 3]))
r2 = r2_score(y_test["SME_As"], pred[:, 3])
mae = mean_absolute_error(y_test["SME_As"], pred[:, 3])
print("================As=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("As")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_As"], pred[:, 3])
xmin = min(min(y_test["SME_As"]), min(pred[:, 3])) - 10
xmax = max(max(y_test["SME_As"]), max(pred[:, 3])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred[:, 4]))
r2 = r2_score(y_test["Hysteresis"], pred[:, 4])
mae = mean_absolute_error(y_test["Hysteresis"], pred[:, 4])
print("=============Hysteresis==============")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred[:, 4])
xmin = min(min(y_test["Hysteresis"]), min(pred[:, 4])) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred[:, 4])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train_filtered.shape[1] - 2065


### Trial with 22 selected features

In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=22,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["SME_Ms"], pred[:, 0]))
r2 = r2_score(y_test["SME_Ms"], pred[:, 0])
mae = mean_absolute_error(y_test["SME_Ms"], pred[:, 0])
print("================Ms=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Ms")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Ms"], pred[:, 0])
xmin = min(min(y_test["SME_Ms"]), min(pred[:, 0])) - 10
xmax = max(max(y_test["SME_Ms"]), max(pred[:, 0])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Mf"], pred[:, 1]))
r2 = r2_score(y_test["SME_Mf"], pred[:, 1])
mae = mean_absolute_error(y_test["SME_Mf"], pred[:, 1])
print("================Mf=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Mf")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Mf"], pred[:, 1])
xmin = min(min(y_test["SME_Mf"]), min(pred[:, 1])) - 10
xmax = max(max(y_test["SME_Mf"]), max(pred[:, 1])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Af"], pred[:, 2]))
r2 = r2_score(y_test["SME_Af"], pred[:, 2])
mae = mean_absolute_error(y_test["SME_Af"], pred[:, 2])
print("================Af=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Af")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Af"], pred[:, 2])
xmin = min(min(y_test["SME_Af"]), min(pred[:, 2])) - 10
xmax = max(max(y_test["SME_Af"]), max(pred[:, 2])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_As"], pred[:, 3]))
r2 = r2_score(y_test["SME_As"], pred[:, 3])
mae = mean_absolute_error(y_test["SME_As"], pred[:, 3])
print("================As=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("As")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_As"], pred[:, 3])
xmin = min(min(y_test["SME_As"]), min(pred[:, 3])) - 10
xmax = max(max(y_test["SME_As"]), max(pred[:, 3])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred[:, 4]))
r2 = r2_score(y_test["Hysteresis"], pred[:, 4])
mae = mean_absolute_error(y_test["Hysteresis"], pred[:, 4])
print("=============Hysteresis==============")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred[:, 4])
xmin = min(min(y_test["Hysteresis"]), min(pred[:, 4])) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred[:, 4])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


Historical feature-selection notes:

```text
['jarvis_avg_X_subs_mol_vol', 'jarvis_avg_atom_rad_divi_therm_cond', 'jarvis_avg_atom_rad_divi_voro_coord', 'jarvis_avg_bp_mult_X', 'jarvis_dev_C-1', 'jarvis_dev_atom_rad_divi_bp', 'jarvis_dev_atom_rad_divi_polzbl', 'jarvis_dev_first_ion_en_subs_voro_coord', 'jarvis_dev_hfus_mult_X', 'jarvis_range_X_subs_bp', 'mat2vec_avg_107', 'mat2vec_dev_153', 'mat2vec_range_53', 'comb_Ni+Pd-Ti', 'comb_Ti/Cu-Ni-Ti', 'comb_Cu/Hf+Ni-Ti', 'comb_Cu/Co+Cu-Hf+Ni-Ti-Zr', 'comb_Zr/Co-Cu+Hf-Ni+Ti', 'comb_Co/Co+Cu-Ni+Pd+Ti', 'comb_Co/Co+Cu-Ni+Ti+Zr', 'comb_Co/Co+Cu+Hf-Ni+Pd+Ti-Zr', 'Processing_FinalHT_Temp']
```

## Enthalpy feature selection

In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap

pd.options.mode.copy_on_write = True


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
outliers = df_lit[(df_lit["Total"] > 100.5) | (df_lit["Total"] < 99.5)]
# Save outliers to CSV
outliers.to_csv("drops_output.csv", index=False)
print(len(outliers))
# Drop outliers from df_lit
df_lit = df_lit[~df_lit.index.isin(outliers.index)]

df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)
df_lit


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :7].columns.to_list()].sum(axis=1)
outliers = df_iter1[(df_iter1["Total"] > 100.5) | (df_iter1["Total"] < 99.5)]
# Save outliers to CSV
outliers.to_csv("drops_output2.csv", index=False)
print(len(outliers))
df_iter1 = df_iter1[~df_iter1.index.isin(outliers.index)]

df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)
df_iter1


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_2_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df_iter2 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]
df_iter2.iloc[:, :7] = df_iter2.iloc[:, :7] * 100
df_iter2.loc[:, "Total"] = df_iter2[df_iter2.iloc[:, :7].columns.to_list()].sum(axis=1)
outliers = df_iter2[(df_iter2["Total"] > 100.5) | (df_iter2["Total"] < 99.5)]
# Save outliers to CSV
outliers.to_csv("drops_output3.csv", index=False)
print(len(outliers))
df_iter2 = df_iter2[~df_iter2.index.isin(outliers.index)]

df_iter2 = df_iter2.drop("Total", axis=1)
df_iter2["Hysteresis"] = df_iter2["SME_Af"] - df_iter2["SME_Ms"]
df_iter2.drop_duplicates(inplace=True)
df_iter2.reset_index(inplace=True, drop=True)
df_iter2


In [ ]:
# Initial data and feature generation
analyzer = FeatureGenerator(df_lit.iloc[:, :40])
analyzer.generate_composition_formula()
main_elements = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
X_features = analyzer.generate_features_all(main_elements=main_elements)
# Concatenate the features and additional columns at once
X_lit = pd.concat([X_features, df_lit.iloc[:, 40:45]], axis=1)
X_lit


In [ ]:
# Ensure all required columns are present in df_iter1
required_columns = [
    "Ag",
    "Al",
    "Au",
    "B",
    "Bi",
    "Ce",
    "Co",
    "Cr",
    "Cu",
    "Dy",
    "Er",
    "Fe",
    "Ga",
    "Gd",
    "Hf",
    "In",
    "La",
    "Mn",
    "Mo",
    "Nb",
    "Nd",
    "Ni",
    "Pb",
    "Pd",
    "Pr",
    "Pt",
    "Re",
    "Rh",
    "Sb",
    "Sc",
    "Si",
    "Sn",
    "Ta",
    "Te",
    "Ti",
    "Tl",
    "V",
    "W",
    "Y",
    "Zr",
]
for column in required_columns:
    if column not in df_iter1.columns:
        df_iter1[column] = 0

# Select the required columns
df_iter1_elnames = df_iter1[required_columns]

# Initial data and feature generation
analyzer = FeatureGenerator(df_iter1_elnames)
analyzer.generate_composition_formula()
main_elements = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
X_features = analyzer.generate_features_all(main_elements=main_elements)
X_iter1 = pd.concat([X_features, df_iter1.iloc[:, 7:12]], axis=1)
X_iter1


In [ ]:
# Ensure all required columns are present in df_iter2
required_columns = [
    "Ag",
    "Al",
    "Au",
    "B",
    "Bi",
    "Ce",
    "Co",
    "Cr",
    "Cu",
    "Dy",
    "Er",
    "Fe",
    "Ga",
    "Gd",
    "Hf",
    "In",
    "La",
    "Mn",
    "Mo",
    "Nb",
    "Nd",
    "Ni",
    "Pb",
    "Pd",
    "Pr",
    "Pt",
    "Re",
    "Rh",
    "Sb",
    "Sc",
    "Si",
    "Sn",
    "Ta",
    "Te",
    "Ti",
    "Tl",
    "V",
    "W",
    "Y",
    "Zr",
]
for column in required_columns:
    if column not in df_iter2.columns:
        df_iter2[column] = 0

# Select the required columns
df_iter2_elnames = df_iter2[required_columns]

# Initial data and feature generation
analyzer = FeatureGenerator(df_iter2_elnames)
analyzer.generate_composition_formula()
main_elements = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
X_features = analyzer.generate_features_all(main_elements=main_elements)
X_iter2 = pd.concat([X_features, df_iter2.iloc[:, 7:12]], axis=1)
X_iter2


In [ ]:
X_train = pd.concat([X_lit, X_iter1], axis=0).reset_index(drop=True)
y_train = pd.concat([df_lit[["Enthalpy"]], df_iter1[["Enthalpy"]]], axis=0).reset_index(
    drop=True
)
X_train


In [ ]:
y_train


In [ ]:
X_test = X_iter2.copy()
X_test


In [ ]:
y_test = df_iter2[["Enthalpy"]].copy()
y_test


### Remove highly correlated features

In [ ]:
%%time
# Apply the same correlation threshold before selecting enthalpy descriptors.
import pandas as pd

# Combine X_train and X_test
X_combined = pd.concat([X_train, X_test], axis=0)

# Calculate absolute correlation matrix
correlation_matrix = X_combined.corr().abs()

# Select the upper triangle of the correlation matrix
upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)

# Find features to drop (correlation > 0.9)
features_to_drop = [
    column for column in upper_triangle.columns if any(upper_triangle[column] > 0.9)
]

# Drop identified features from X_train and X_test
X_train_filtered = X_train.drop(columns=features_to_drop)
X_test_filtered = X_test.drop(columns=features_to_drop)

# Save the remaining features in X_train_filtered to P_C_features.csv
X_train_filtered.columns.to_series().to_csv("P_C_features_enthalpy.csv", index=False)

# Return the filtered datasets
X_train_filtered, X_test_filtered


In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


print("Performance")
y_test = np.array(y_test).astype(float)
pred = np.array(pred).astype(float)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print("================Enthalpy=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test, pred)
xmin = min(min(y_test), min(pred)) - 3
xmax = max(max(y_test), max(pred)) + 3
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train_filtered.shape[1] - 1915


In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=3,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


print("Performance")
y_test = np.array(y_test).astype(float)
pred = np.array(pred).astype(float)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print("================Enthalpy=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test, pred)
xmin = min(min(y_test), min(pred)) - 3
xmax = max(max(y_test), max(pred)) + 3
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train_filtered.shape[1] - 1909


In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=9,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


print("Performance")
y_test = np.array(y_test).astype(float)
pred = np.array(pred).astype(float)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print("================Enthalpy=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test, pred)
xmin = min(min(y_test), min(pred)) - 3
xmax = max(max(y_test), max(pred)) + 3
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=50,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Print performance metrics
print("Performance")
y_test = np.array(y_test).astype(float)
pred = np.array(pred).astype(float)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print("================Enthalpy=================")
print("RMSE: {} \nMAE: {} \nR2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3)))

# Plotting actual vs predicted values
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test, pred)

# Determine plot limits
xmin = min(np.min(y_test), np.min(pred)) - 3
xmax = max(np.max(y_test), np.max(pred)) + 3

# Set axis limits and plot y=x reference line
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

plt.show()


In [ ]:
X_train_filtered.shape[1] - 1912


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Print performance metrics
print("Performance")
y_test = np.array(y_test).astype(float)
pred = np.array(pred).astype(float)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print("================Enthalpy=================")
print("RMSE: {} \nMAE: {} \nR2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3)))

# Plotting actual vs predicted values
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test, pred)

# Determine plot limits
xmin = min(np.min(y_test), np.min(pred)) - 3
xmax = max(np.max(y_test), np.max(pred)) + 3

# Set axis limits and plot y=x reference line
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

plt.show()


In [ ]:
%%time
dtrain = cb.Pool(X_train_filtered, label=y_train)
dvalid = cb.Pool(X_test_filtered, label=y_test)
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
    "iterations": 1500,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=50,
    features_for_select=f"0-{X_train_filtered.shape[1] - 1}",
    num_features_to_select=6,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test_filtered)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train_filtered.shape[1] - corresponding_feature_count
    )
)


import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Print performance metrics
print("Performance")
y_test = np.array(y_test).astype(float)
pred = np.array(pred).astype(float)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)
mae = mean_absolute_error(y_test, pred)
print("================Enthalpy=================")
print("RMSE: {} \nMAE: {} \nR2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3)))

# Plotting actual vs predicted values
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test, pred)

# Determine plot limits
xmin = min(np.min(y_test), np.min(pred)) - 3
xmax = max(np.max(y_test), np.max(pred)) + 3

# Set axis limits and plot y=x reference line
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

plt.show()


Historical feature-selection notes:

```text
Best result for enthalpy: ['mat2vec_dev_33', 'mat2vec_dev_48', 'mat2vec_dev_92', 'onehot_max_115', 'comb_Cu/Co+Cu-Hf+Ni-Ti-Zr', 'comb_Hf/Co-Cu-Ni+Pd+Ti+Zr']
```

### Early-stopping trials

Historical validation curves plateaued or fluctuated after additional iterations.
The following trials compare feature-selection settings and stopping patience.

## Additional feature-selection trials

The following cells record alternative feature counts and stopping settings.
Each training cell replaces `model` and `summary`; run its evaluation cells
immediately afterwards to keep predictions paired with the intended model.

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train)
dvalid = cb.Pool(X_test, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 30,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
model.get_all_params()


Historical observation: `od_wait=50` performed better than `od_wait=30` in this trial.

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train)
dvalid = cb.Pool(X_test, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
model.get_all_params()


In [ ]:
X_train.shape[1] - 5805


### Trial with 10 selected features

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train)
dvalid = cb.Pool(X_test, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=10,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
model.get_all_params()


In [ ]:
X_train.shape[1] - 5779


### Compare a larger feature subset

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train)
dvalid = cb.Pool(X_test, label=y_test)
params = {
    "loss_function": "MultiRMSE",
    "eval_metric": "MultiRMSE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=36,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
model.get_all_params()


In [ ]:
X_train.shape[1] - 5779


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["SME_Ms"], pred[:, 0]))
r2 = r2_score(y_test["SME_Ms"], pred[:, 0])
mae = mean_absolute_error(y_test["SME_Ms"], pred[:, 0])
print("================Ms=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Ms")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Ms"], pred[:, 0])
xmin = min(min(y_test["SME_Ms"]), min(pred[:, 0])) - 10
xmax = max(max(y_test["SME_Ms"]), max(pred[:, 0])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Mf"], pred[:, 1]))
r2 = r2_score(y_test["SME_Mf"], pred[:, 1])
mae = mean_absolute_error(y_test["SME_Mf"], pred[:, 1])
print("================Mf=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Mf")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Mf"], pred[:, 1])
xmin = min(min(y_test["SME_Mf"]), min(pred[:, 1])) - 10
xmax = max(max(y_test["SME_Mf"]), max(pred[:, 1])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_Af"], pred[:, 2]))
r2 = r2_score(y_test["SME_Af"], pred[:, 2])
mae = mean_absolute_error(y_test["SME_Af"], pred[:, 2])
print("================Af=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Af")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Af"], pred[:, 2])
xmin = min(min(y_test["SME_Af"]), min(pred[:, 2])) - 10
xmax = max(max(y_test["SME_Af"]), max(pred[:, 2])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")

rmse = np.sqrt(mean_squared_error(y_test["SME_As"], pred[:, 3]))
r2 = r2_score(y_test["SME_As"], pred[:, 3])
mae = mean_absolute_error(y_test["SME_As"], pred[:, 3])
print("================As=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("As")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_As"], pred[:, 3])
xmin = min(min(y_test["SME_As"]), min(pred[:, 3])) - 10
xmax = max(max(y_test["SME_As"]), max(pred[:, 3])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred[:, 4]))
r2 = r2_score(y_test["Hysteresis"], pred[:, 4])
mae = mean_absolute_error(y_test["Hysteresis"], pred[:, 4])
print("=============Hysteresis==============")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred[:, 4])
xmin = min(min(y_test["Hysteresis"]), min(pred[:, 4])) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred[:, 4])) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


Historical recorded metrics (not rerun during cleanup):

```text
Performance
================Ms=================
RMSE: 56.314 
 MAE: 46.91 
 R2: 0.47
================Mf=================
RMSE: 49.008 
 MAE: 39.979 
 R2: 0.575
================Af=================
RMSE: 106.728 
 MAE: 90.176 
 R2: 0.539
================As=================
RMSE: 110.629 
 MAE: 87.538 
 R2: 0.504
=============Hysteresis==============
RMSE: 84.422 
 MAE: 52.319 
 R2: 0.194
```

### Fit Ms as a single target

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train[["SME_Ms"]])
dvalid = cb.Pool(X_test, label=y_test[["SME_Ms"]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["SME_Ms"], pred))
r2 = r2_score(y_test["SME_Ms"], pred)
mae = mean_absolute_error(y_test["SME_Ms"], pred)
print("================Ms=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Ms")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Ms"], pred)
xmin = min(min(y_test["SME_Ms"]), min(pred)) - 10
xmax = max(max(y_test["SME_Ms"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train.shape[1] - 5805


### Compare a larger Ms feature subset

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train[["SME_Ms"]])
dvalid = cb.Pool(X_test, label=y_test[["SME_Ms"]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=10,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["SME_Ms"], pred))
r2 = r2_score(y_test["SME_Ms"], pred)
mae = mean_absolute_error(y_test["SME_Ms"], pred)
print("================Ms=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Ms")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["SME_Ms"], pred)
xmin = min(min(y_test["SME_Ms"]), min(pred)) - 10
xmax = max(max(y_test["SME_Ms"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


### Fit hysteresis as a single target

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train[["Hysteresis"]])
dvalid = cb.Pool(X_test, label=y_test[["Hysteresis"]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred))
r2 = r2_score(y_test["Hysteresis"], pred)
mae = mean_absolute_error(y_test["Hysteresis"], pred)
print("================Hysteresis=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred)
xmin = min(min(y_test["Hysteresis"]), min(pred)) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train.shape[1] - 5807


### Hysteresis trial with 1,300 iterations and more features

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train[["Hysteresis"]])
dvalid = cb.Pool(X_test, label=y_test[["Hysteresis"]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
    "iterations": 1300,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=8,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred))
r2 = r2_score(y_test["Hysteresis"], pred)
mae = mean_absolute_error(y_test["Hysteresis"], pred)
print("================Hysteresis=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred)
xmin = min(min(y_test["Hysteresis"]), min(pred)) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train.shape[1] - 5703


### Compare a larger hysteresis feature subset

In [ ]:
%%time
dtrain = cb.Pool(X_train, label=y_train[["Hysteresis"]])
dvalid = cb.Pool(X_test, label=y_test[["Hysteresis"]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
    "iterations": 1300,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=112,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary["selected_features_names"]


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred))
r2 = r2_score(y_test["Hysteresis"], pred)
mae = mean_absolute_error(y_test["Hysteresis"], pred)
print("================Hysteresis=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred)
xmin = min(min(y_test["Hysteresis"]), min(pred)) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


### Fit Mf with eight selected features

In [ ]:
%%time
study_target = "SME_Mf"


dtrain = cb.Pool(X_train, label=y_train[[study_target]])
dvalid = cb.Pool(X_test, label=y_test[[study_target]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=1,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)

print("Performance")
rmse = np.sqrt(mean_squared_error(y_test[study_target], pred))
r2 = r2_score(y_test[study_target], pred)
mae = mean_absolute_error(y_test[study_target], pred)
print(f"================{study_target}=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title(study_target)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test[study_target], pred)
xmin = min(min(y_test[study_target]), min(pred)) - 10
xmax = max(max(y_test[study_target]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


In [ ]:
X_train.shape[1] - 5807


### Compare a larger Mf feature subset

In [ ]:
%%time
study_target = "SME_Mf"


dtrain = cb.Pool(X_train, label=y_train[[study_target]])
dvalid = cb.Pool(X_test, label=y_test[[study_target]])
params = {
    "loss_function": "MAE",
    "eval_metric": "MAE",
    "verbose": False,
    "use_best_model": True,
    "od_type": "Iter",
    "od_wait": 50,
}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=30,
    features_for_select=f"0-{X_train.shape[1] - 1}",
    num_features_to_select=8,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)

print(summary["selected_features_names"])

import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print(
    "Number of features to select: {}".format(
        X_train.shape[1] - corresponding_feature_count
    )
)

print("Performance")
rmse = np.sqrt(mean_squared_error(y_test[study_target], pred))
r2 = r2_score(y_test[study_target], pred)
mae = mean_absolute_error(y_test[study_target], pred)
print(f"================{study_target}=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title(study_target)
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test[study_target], pred)
xmin = min(min(y_test[study_target]), min(pred)) - 10
xmax = max(max(y_test[study_target]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


### Hysteresis trial notes

In [ ]:
summary


Historical recorded metrics (not rerun during cleanup):

```text
Performance
================Hysteresis=================
RMSE: 44.748 
 MAE: 28.165 
 R2: -0.024
```

### Additional hysteresis trial

In [ ]:
rows_with_zero_or_nan = y_train[
    (y_train["Hysteresis"] == 0) | (y_train["Hysteresis"].isna())
]

print(rows_with_zero_or_nan)


In [ ]:
dtrain = cb.Pool(X_train, label=y_train[["Hysteresis"]])
dvalid = cb.Pool(X_test, label=y_test[["Hysteresis"]])
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=10,
    features_for_select="0-5815",
    num_features_to_select=47,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print("Number of features to select: {}".format(5816 - corresponding_feature_count))


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["Hysteresis"], pred))
r2 = r2_score(y_test["Hysteresis"], pred)
mae = mean_absolute_error(y_test["Hysteresis"], pred)
print("================Hysteresis=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Hysteresis")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Hysteresis"], pred)
xmin = min(min(y_test["Hysteresis"]), min(pred)) - 10
xmax = max(max(y_test["Hysteresis"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


## Enthalpy model fitting

In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap

pd.options.mode.copy_on_write = True


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output3.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)
df_lit


In [ ]:
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]
df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output4.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)
df_iter1


In [ ]:
y_train = df_lit.iloc[:, -6:]
y_train


In [ ]:
y_test = df_iter1.iloc[:, -6:]
y_test


In [ ]:
analyzer = FeatureGenerator(df_lit.iloc[:, :40])
analyzer.generate_composition_formula()
analyzer.generate_features_all()


In [ ]:
# Initial data and feature generation
analyzer = FeatureGenerator(df_lit.iloc[:, :40])
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()

# Concatenate the features and additional columns at once
X_train = pd.concat([X_features, df_lit.iloc[:, 40:45]], axis=1)

# Create a de-fragmented copy of X_train
X_train = X_train.copy()

# Apply the transformation
lambda2_model = Lambda2Model(X_train)
X_train = lambda2_model.predict_transformation_and_lambda2()

# Create a de-fragmented copy of X_train after transformation
X_train = X_train.copy()
X_train = X_train.drop(columns=["Predicted_Transformation_Type"])
X_train


In [ ]:
# Ensure all required columns are present in df_iter1
required_columns = [
    "Ag",
    "Al",
    "Au",
    "B",
    "Bi",
    "Ce",
    "Co",
    "Cr",
    "Cu",
    "Dy",
    "Er",
    "Fe",
    "Ga",
    "Gd",
    "Hf",
    "In",
    "La",
    "Mn",
    "Mo",
    "Nb",
    "Nd",
    "Ni",
    "Pb",
    "Pd",
    "Pr",
    "Pt",
    "Re",
    "Rh",
    "Sb",
    "Sc",
    "Si",
    "Sn",
    "Ta",
    "Te",
    "Ti",
    "Tl",
    "V",
    "W",
    "Y",
    "Zr",
]
for column in required_columns:
    if column not in df_iter1.columns:
        df_iter1[column] = 0

# Select the required columns
df_iter1_elnames = df_iter1[required_columns]

# Initial data and feature generation
analyzer = FeatureGenerator(df_iter1_elnames)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()

# Concatenate the features and additional columns at once
X_test = pd.concat([X_features, df_iter1.iloc[:, 7:12]], axis=1)

# Create a de-fragmented copy of X_test
X_test = X_test.copy()

# Apply the transformation
lambda2_model = Lambda2Model(X_test)
X_test = lambda2_model.predict_transformation_and_lambda2()

# Create a de-fragmented copy of X_train after transformation
X_test = X_test.copy()
X_test = X_test.drop(columns=["Predicted_Transformation_Type"])

X_test


In [ ]:
rows_with_zero_or_nan = y_train[
    (y_train["Enthalpy"] == 0) | (y_train["Enthalpy"].isna())
]

print(rows_with_zero_or_nan)


In [ ]:
5816 - 5810


In [ ]:
dtrain = cb.Pool(X_train, label=y_train[["Enthalpy"]])
dvalid = cb.Pool(X_test, label=y_test[["Enthalpy"]])
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
summary = model.select_features(
    dtrain,
    eval_set=dvalid,
    steps=10,
    features_for_select="0-5815",
    num_features_to_select=6,
    algorithm="RecursiveByShapValues",
    shap_calc_type="Exact",
    train_final_model=True,
    logging_level="Silent",
    plot=True,
)

pred = model.predict(X_test)


In [ ]:
summary


In [ ]:
import matplotlib.pyplot as plt

# Assuming summary is already defined
loss_values = summary["loss_graph"]["loss_values"]
removed_features_count = summary["loss_graph"]["removed_features_count"]

# Plotting the graph
plt.plot(removed_features_count, loss_values, picker=True)
plt.xlabel("Removed Features Count")
plt.ylabel("Loss Values")
plt.title("Loss Values vs. Removed Features Count")
plt.show()

# Finding the minimum loss value and corresponding feature count
min_loss_value = min(loss_values)
min_loss_index = loss_values.index(min_loss_value)
corresponding_feature_count = removed_features_count[min_loss_index]

print(
    f"The minimum loss value is {min_loss_value} at {corresponding_feature_count} removed features."
)
print("Number of features to select: {}".format(5816 - corresponding_feature_count))


In [ ]:
print("Performance")
rmse = np.sqrt(mean_squared_error(y_test["Enthalpy"], pred))
r2 = r2_score(y_test["Enthalpy"], pred)
mae = mean_absolute_error(y_test["Enthalpy"], pred)
print("================Enthalpy=================")
print(
    "RMSE: {} \n MAE: {} \n R2: {}".format(round(rmse, 3), round(mae, 3), round(r2, 3))
)
plt.figure(figsize=(7, 7))
plt.title("Enthalpy")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.scatter(y_test["Enthalpy"], pred)
xmin = min(min(y_test["Enthalpy"]), min(pred)) - 10
xmax = max(max(y_test["Enthalpy"]), max(pred)) + 10
plt.xlim(xmin, xmax)
plt.ylim(xmin, xmax)
plt.plot([xmin, xmax], [xmin, xmax], color="red")


## Train the screening model

In [ ]:
import sys
from importlib.metadata import version

import ast
import pandas as pd
import numpy as np


from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Python version: " + sys.version)
print("NumPy version: {}".format(version("numpy")))
print("pandas version: {}".format(version("pandas")))
print("CBFV version: {}".format(version("CBFV")))


In [ ]:
# Combine the literature and first iteration to train the candidate-screening model.
# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["SME_Ms"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_atom_mass_mult_voro_coord",
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_first_ion_en_subs_polzbl",
        "jarvis_avg_hfus_divi_voro_coord",
        "jarvis_avg_mp_add_atom_rad",
        "jarvis_avg_polzbl_divi_therm_cond",
        "jarvis_avg_polzbl_mult_atom_rad",
        "jarvis_avg_therm_cond_divi_polzbl",
        "jarvis_dev_X_subs_atom_rad",
        "jarvis_dev_atom_mass_mult_atom_rad",
        "jarvis_dev_bp_divi_polzbl",
        "jarvis_dev_mol_vol_subs_polzbl",
        "magpie_avg_CovalentRadius",
        "magpie_avg_NUnfilled",
        "oliynyk_dev_Pauling_Electronegativity",
        "oliynyk_dev_gilmor_number_of_valence_electron",
        "mat2vec_sum_7",
        "mat2vec_sum_154",
        "mat2vec_avg_7",
        "mat2vec_avg_77",
        "mat2vec_avg_84",
        "mat2vec_avg_94",
        "mat2vec_dev_35",
        "mat2vec_dev_52",
        "mat2vec_dev_63",
        "mat2vec_dev_73",
        "mat2vec_dev_97",
        "mat2vec_dev_121",
        "mat2vec_dev_124",
        "mat2vec_dev_154",
        "mat2vec_dev_166",
        "Processing_FinalHT_Time",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


## Candidate screening

Use the preceding fitted model to predict candidate properties in chunks. The
paths in the configuration cell identify the external composition spaces and
previous screening results.

In [ ]:
# Generate descriptors and predict Ms in bounded batches using the fitted model.
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.copy_on_write = True


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Configuration to avoid modifying the original data
pd.options.mode.copy_on_write = True

chunk_size = 800000  # Process 800,000 rows at a time


# Function to process a single chunk
def process_chunk(chunk):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in chunk.columns:
            chunk[element] = 0.0
    analyzer = FeatureGenerator(chunk)
    comp_df = analyzer.generate_composition_formula()
    chunk = analyzer.generate_features_all()
    # Set processing conditions for each chunk
    chunk["Processing_BPHT_Temp"] = 20.0
    chunk["Processing_BPHT_Time"] = 0.0
    chunk["Processing_Rolling_Temp"] = 20.0
    chunk["Processing_HR_Red"] = 0.0
    chunk["Processing_CR_Red"] = 0.0
    chunk["Processing_Extrusion_Temp"] = 20.0
    chunk["Processing_Extrusion_Area_Reduction(%)"] = 0.0
    chunk["Processing_ECAE_Temp"] = 20.0
    chunk["Processing_ECAE_Route"] = 0.0
    chunk["Processing_APHT_Temp"] = 1100.0
    chunk["Processing_APHT_Time"] = 24.0
    chunk["SME_Test_Applied_Stress(MPa)"] = 0.0
    chunk["SME_Test_Cycle"] = 2.0

    chunk["Processing_FinalHT_Temp"] = temp
    chunk["Processing_FinalHT_Time"] = time

    chunk = chunk[
        [
            "jarvis_avg_atom_mass_mult_voro_coord",
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_first_ion_en_subs_polzbl",
            "jarvis_avg_hfus_divi_voro_coord",
            "jarvis_avg_mp_add_atom_rad",
            "jarvis_avg_polzbl_divi_therm_cond",
            "jarvis_avg_polzbl_mult_atom_rad",
            "jarvis_avg_therm_cond_divi_polzbl",
            "jarvis_dev_X_subs_atom_rad",
            "jarvis_dev_atom_mass_mult_atom_rad",
            "jarvis_dev_bp_divi_polzbl",
            "jarvis_dev_mol_vol_subs_polzbl",
            "magpie_avg_CovalentRadius",
            "magpie_avg_NUnfilled",
            "oliynyk_dev_Pauling_Electronegativity",
            "oliynyk_dev_gilmor_number_of_valence_electron",
            "mat2vec_sum_7",
            "mat2vec_sum_154",
            "mat2vec_avg_7",
            "mat2vec_avg_77",
            "mat2vec_avg_84",
            "mat2vec_avg_94",
            "mat2vec_dev_35",
            "mat2vec_dev_52",
            "mat2vec_dev_63",
            "mat2vec_dev_73",
            "mat2vec_dev_97",
            "mat2vec_dev_121",
            "mat2vec_dev_124",
            "mat2vec_dev_154",
            "mat2vec_dev_166",
            "Processing_FinalHT_Time",
        ]
    ]

    # Process the chunk with your FeatureGenerator and model prediction

    pred = model.predict(chunk)
    # Combine the predictions with the chunk
    chunk = pd.concat(
        [
            #             chunk[['Ni', 'Ti', 'Cu', 'Hf', 'Zr', 'Pd', 'Co']],
            chunk,
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Ms (C)",
                ],
            ),
        ],
        axis=1,
    )
    return chunk


# Process each chunk and immediately write to file
first_chunk = True

for chunk in pd.read_csv(COMPOSITION_SPACE_CSV, chunksize=chunk_size):
    chunk.reset_index(drop=True, inplace=True)
    processed_chunk = process_chunk(chunk)

    # Determine write mode ('w' for the first chunk to create the file, 'a' to append)
    write_mode = "w" if first_chunk else "a"
    header = first_chunk  # Only write header for the first chunk

    # Write the processed chunk to file
    processed_chunk.to_csv(
        f"q_r_all_{time}_{temp}_ms_pred_added.csv",
        index=False,
        mode=write_mode,
        header=header,
    )

    # After the first chunk has been processed and written, switch off the first_chunk flag
    if first_chunk:
        first_chunk = False


In [ ]:
import pandas as pd

df1 = pd.read_csv(COMPOSITION_SPACE_CSV)
df1


In [ ]:
df2 = pd.read_csv("q_r_all_24_950_ms_pred_added.csv")
df2


In [ ]:
df3 = pd.concat([df1, df2["Predicted Ms (C)"]], axis=1)
df3


In [ ]:
df4 = df3[df3["Predicted Ms (C)"] > 200.0].reset_index(drop=True)
df4.drop_duplicates(inplace=True)
df4.reset_index(inplace=True, drop=True)
df4


In [ ]:
df5 = df4.iloc[:, :7]
df5


In [ ]:
df_prev = pd.read_csv(PREVIOUS_TC_INPUT_CSV).iloc[:, :7]
df_prev


In [ ]:
common_rows = pd.merge(df5, df_prev, how="inner")

# Counting the number of matching rows
num_common_rows = common_rows.shape[0]

print(f"Number of exact rows in df5 that exist in df_prev: {num_common_rows}")


In [ ]:
# Apply the candidate composition bounds before exporting thermodynamic inputs.
import pandas as pd

df1 = pd.read_csv(GENERATED_COMPOSITION_SPACE_CSV)
df2 = pd.read_csv(PREVIOUS_MS_PREDICTIONS_CSV)

df = pd.concat([df1, df2["Predicted Ms (C)"]], axis=1)


new_df = df[
    (df["Predicted Ms (C)"] > 200) & (df["Predicted Ms (C)"] < 400)
].reset_index(drop=True)
new_df = new_df.drop_duplicates().reset_index(drop=True)
new_df = new_df[(new_df["Ni"] >= 20) & (new_df["Ni"] <= 51.5)]
new_df = new_df[
    (new_df["Ti"] >= 20) & (new_df["Ti"] <= 50)
]  # Ti < 50 due to the additional constraint
new_df = new_df[(new_df["Cu"] == 0) | ((new_df["Cu"] >= 5) & (new_df["Cu"] <= 25))]
new_df = new_df[(new_df["Hf"] == 0) | ((new_df["Hf"] >= 5) & (new_df["Hf"] <= 25))]
new_df = new_df[(new_df["Zr"] == 0) | ((new_df["Zr"] >= 5) & (new_df["Zr"] <= 25))]
new_df = new_df[(new_df["Pd"] >= 0) & (new_df["Pd"] <= 25)]
new_df = new_df[(new_df["Co"] >= 0) & (new_df["Co"] <= 5)]

# # Apply the sum constraints
new_df = new_df[
    (new_df["Ti"] + new_df["Hf"] + new_df["Zr"] >= 48.5)
    & (new_df["Ti"] + new_df["Hf"] + new_df["Zr"] <= 50)
]
new_df = new_df[
    (new_df["Ni"] + new_df["Cu"] + new_df["Co"] + new_df["Pd"] >= 48.5)
    & (new_df["Ni"] + new_df["Cu"] + new_df["Co"] + new_df["Pd"] <= 51.5)
]
new_df = new_df[
    (
        new_df["Ni"]
        + new_df["Ti"]
        + new_df["Cu"]
        + new_df["Hf"]
        + new_df["Zr"]
        + new_df["Pd"]
        + new_df["Co"]
        <= 100.1
    )
    & (
        new_df["Ni"]
        + new_df["Ti"]
        + new_df["Cu"]
        + new_df["Hf"]
        + new_df["Zr"]
        + new_df["Pd"]
        + new_df["Co"]
        >= 99.9
    )
]
new_df
new_df.to_csv("tc/file.csv", index=False)


In [ ]:
new_df


## Thermodynamic batch jobs

This cell reads `template.py` and `template.sh` from the working directory,
creates a pair of scripts per chunk, and submits each job with `sbatch`.
Place the selected compositions in `file.csv` in that directory before running
the jobs. The prior screening cell writes `tc/file.csv`; copy that file into the
job working directory if these sections run from different locations. Configure
the shell template for your cluster and set `TC_RESULTS_DIRECTORY` before
collecting results. Thermo-Calc and its Python API are required for these jobs.

In [ ]:
import math
import subprocess
import time  # Import the time module for sleep functionality


total_samples = 45373
samples_per_chunk = 1500
num_chunks = math.ceil(total_samples / samples_per_chunk)
print(num_chunks)
python_template_filename = "template.py"  # Name of your Python template
bash_template_filename = "template.sh"  # Name of your bash script template


for chunk in range(num_chunks):
    start_idx = chunk * samples_per_chunk
    end_idx = min((chunk + 1) * samples_per_chunk, total_samples)

    # Generate Python file name for the current chunk
    python_filename = f"tc_chunk_{chunk}.py"

    # Read and adjust the Python template content
    with open(python_template_filename, "r") as file:
        python_content = file.read()

    # Replace placeholders with actual values
    python_content = python_content.replace(
        'composition_df = pd.read_csv("file.csv")',
        f'composition_df = pd.read_csv("file.csv").iloc[{start_idx}:{end_idx}, 0:9].reset_index(drop=True)',
    )
    python_content = python_content.replace(
        '"tcresults_ct.csv"', f'"tcresults_ct_chunk_{chunk}.csv"'
    )

    # Write the modified content to the new Python file
    with open(python_filename, "w") as file:
        file.write(python_content)

    # Generate bash script name for the current chunk
    bash_filename = f"run_chunk_{chunk}.sh"

    # Read and adjust the bash template content
    with open(bash_template_filename, "r") as file:
        bash_content = file.read()

    # Adjust the bash script to use the current Python file
    bash_content = bash_content.replace(
        "python filname.py", f"python {python_filename}"
    )
    # Optionally, adjust job name to reflect the chunk
    bash_content = bash_content.replace(
        "#SBATCH -J tcrun", f"#SBATCH -J tcrun_chunk_{chunk}"
    )

    # Write the modified content to the new bash file
    with open(bash_filename, "w") as file:
        file.write(bash_content)

    # Submit the job immediately after creating the bash script
    try:
        subprocess.run(["sbatch", bash_filename], check=True)
        print(f"Successfully submitted {bash_filename}")
    except subprocess.CalledProcessError as e:
        print(f"Failed to submit {bash_filename}: {e}")

    time.sleep(2)  # Sleep for 2 seconds before submitting the next job

print(
    f"Generated and submitted {num_chunks} Python and bash script files for each chunk, with a 2-second pause between submissions."
)


In [ ]:
# Collect existing per-chunk results from the configured thermodynamic output directory.
import pandas as pd
import os

# Initialize an empty list to store the DataFrames
dfs = []


for i in range(47):
    print(i)
    # Construct the file path
    file_path = TC_RESULTS_DIRECTORY / f"tcresults_ct_chunk_{i}.csv"

    # Check if the file exists
    if os.path.isfile(file_path):
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        print(len(df))

        # Append the DataFrame to the list
        dfs.append(df)
    else:
        print(f"File {file_path} does not exist.")

# Concatenate all DataFrames in the list into a single DataFrame
combined_df = pd.concat(dfs, ignore_index=True)

combined_df


In [ ]:
# Parse saved phase lists as Python literals before applying phase constraints.
import ast  # For converting strings to Python objects

# Convert strings to lists
combined_df["stable_phases"] = combined_df["stable_phases"].apply(ast.literal_eval)

# Get unique phases
unique_phases = set(
    phase for phases in combined_df["stable_phases"] for phase in phases
)
print(list(unique_phases))


In [ ]:
ex_lst = [
    "LIQUID#3",
    "LIQUID#1",
    "LIQUID#2",
    "FCC_L12#1",
    "FCC_L12#2",
    "NI5ZR#1",
    "H_L21#1",
    "SIGMA#1",
    "NI3TI_D024#1",
    "FCC_L12#2",
    "C14_LAVES#1",
    "H_L21#2",
]

# Convert the list to a set for faster membership checking
phase_set = set(ex_lst)

# Filter the DataFrame
new_df = combined_df[
    ~combined_df["stable_phases"].apply(
        lambda x: any(phase in phase_set for phase in x)
    )
]
new_df.reset_index(drop=True, inplace=True)
new_df


In [ ]:
# Get unique phases
unique_phases = set(phase for phases in new_df["stable_phases"] for phase in phases)
print(list(unique_phases))


In [ ]:
new_df.loc[:, "stable_phases"] = new_df["stable_phases"].apply(lambda x: str(x))
new_df = new_df.drop_duplicates().reset_index(drop=True)
new_df = new_df[(new_df["Ni"] >= 20) & (new_df["Ni"] <= 51.5)]
new_df = new_df[
    (new_df["Ti"] >= 20) & (new_df["Ti"] <= 50)
]  # Ti < 50 due to the additional constraint
new_df = new_df[(new_df["Cu"] == 0) | ((new_df["Cu"] >= 5) & (new_df["Cu"] <= 25))]
new_df = new_df[(new_df["Hf"] == 0) | ((new_df["Hf"] >= 5) & (new_df["Hf"] <= 25))]
new_df = new_df[(new_df["Zr"] == 0) | ((new_df["Zr"] >= 5) & (new_df["Zr"] <= 25))]
new_df = new_df[(new_df["Pd"] >= 0) & (new_df["Pd"] <= 25)]
new_df = new_df[(new_df["Co"] >= 0) & (new_df["Co"] <= 5)]

# # Apply the sum constraints
new_df = new_df[
    (new_df["Ti"] + new_df["Hf"] + new_df["Zr"] >= 48.5)
    & (new_df["Ti"] + new_df["Hf"] + new_df["Zr"] <= 50)
]
new_df = new_df[
    (new_df["Ni"] + new_df["Cu"] + new_df["Co"] + new_df["Pd"] >= 48.5)
    & (new_df["Ni"] + new_df["Cu"] + new_df["Co"] + new_df["Pd"] <= 51.5)
]
new_df = new_df[
    (
        new_df["Ni"]
        + new_df["Ti"]
        + new_df["Cu"]
        + new_df["Hf"]
        + new_df["Zr"]
        + new_df["Pd"]
        + new_df["Co"]
        <= 100.1
    )
    & (
        new_df["Ni"]
        + new_df["Ti"]
        + new_df["Cu"]
        + new_df["Hf"]
        + new_df["Zr"]
        + new_df["Pd"]
        + new_df["Co"]
        >= 99.9
    )
]

new_df.reset_index(drop=True, inplace=True)
new_df.to_csv("whole_space_950C_tc_included.csv", index=False)
new_df


## Sequential property predictions

Fit the Ms, hysteresis, and enthalpy models and append their predictions to the
candidate tables. Each block retains its original target, feature list, and
processing conditions; run the matching export cell after the prediction block.

In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["SME_Ms"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_atom_mass_mult_voro_coord",
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_first_ion_en_subs_polzbl",
        "jarvis_avg_hfus_divi_voro_coord",
        "jarvis_avg_mp_add_atom_rad",
        "jarvis_avg_polzbl_divi_therm_cond",
        "jarvis_avg_polzbl_mult_atom_rad",
        "jarvis_avg_therm_cond_divi_polzbl",
        "jarvis_dev_X_subs_atom_rad",
        "jarvis_dev_atom_mass_mult_atom_rad",
        "jarvis_dev_bp_divi_polzbl",
        "jarvis_dev_mol_vol_subs_polzbl",
        "magpie_avg_CovalentRadius",
        "magpie_avg_NUnfilled",
        "oliynyk_dev_Pauling_Electronegativity",
        "oliynyk_dev_gilmor_number_of_valence_electron",
        "mat2vec_sum_7",
        "mat2vec_sum_154",
        "mat2vec_avg_7",
        "mat2vec_avg_77",
        "mat2vec_avg_84",
        "mat2vec_avg_94",
        "mat2vec_dev_35",
        "mat2vec_dev_52",
        "mat2vec_dev_63",
        "mat2vec_dev_73",
        "mat2vec_dev_97",
        "mat2vec_dev_121",
        "mat2vec_dev_124",
        "mat2vec_dev_154",
        "mat2vec_dev_166",
        "Processing_FinalHT_Time",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("whole_space_950C_tc_included.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    X["Processing_FinalHT_Temp"] = temp
    X["Processing_FinalHT_Time"] = time
    X = X[
        [
            "jarvis_avg_atom_mass_mult_voro_coord",
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_first_ion_en_subs_polzbl",
            "jarvis_avg_hfus_divi_voro_coord",
            "jarvis_avg_mp_add_atom_rad",
            "jarvis_avg_polzbl_divi_therm_cond",
            "jarvis_avg_polzbl_mult_atom_rad",
            "jarvis_avg_therm_cond_divi_polzbl",
            "jarvis_dev_X_subs_atom_rad",
            "jarvis_dev_atom_mass_mult_atom_rad",
            "jarvis_dev_bp_divi_polzbl",
            "jarvis_dev_mol_vol_subs_polzbl",
            "magpie_avg_CovalentRadius",
            "magpie_avg_NUnfilled",
            "oliynyk_dev_Pauling_Electronegativity",
            "oliynyk_dev_gilmor_number_of_valence_electron",
            "mat2vec_sum_7",
            "mat2vec_sum_154",
            "mat2vec_avg_7",
            "mat2vec_avg_77",
            "mat2vec_avg_84",
            "mat2vec_avg_94",
            "mat2vec_dev_35",
            "mat2vec_dev_52",
            "mat2vec_dev_63",
            "mat2vec_dev_73",
            "mat2vec_dev_97",
            "mat2vec_dev_121",
            "mat2vec_dev_124",
            "mat2vec_dev_154",
            "mat2vec_dev_166",
            "Processing_FinalHT_Time",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Ms (C)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
processed_df


In [ ]:
pd.concat([processed_df, df[["stable_phases"]]], axis=1).to_csv(
    "whole_space_950C_tc_included_pred_added.csv", index=False
)


In [ ]:
import pandas as pd

df = pd.read_csv("whole_space_950C_tc_included_pred_added.csv")
df


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["SME_Ms"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_atom_mass_mult_voro_coord",
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_first_ion_en_subs_polzbl",
        "jarvis_avg_hfus_divi_voro_coord",
        "jarvis_avg_mp_add_atom_rad",
        "jarvis_avg_polzbl_divi_therm_cond",
        "jarvis_avg_polzbl_mult_atom_rad",
        "jarvis_avg_therm_cond_divi_polzbl",
        "jarvis_dev_X_subs_atom_rad",
        "jarvis_dev_atom_mass_mult_atom_rad",
        "jarvis_dev_bp_divi_polzbl",
        "jarvis_dev_mol_vol_subs_polzbl",
        "magpie_avg_CovalentRadius",
        "magpie_avg_NUnfilled",
        "oliynyk_dev_Pauling_Electronegativity",
        "oliynyk_dev_gilmor_number_of_valence_electron",
        "mat2vec_sum_7",
        "mat2vec_sum_154",
        "mat2vec_avg_7",
        "mat2vec_avg_77",
        "mat2vec_avg_84",
        "mat2vec_avg_94",
        "mat2vec_dev_35",
        "mat2vec_dev_52",
        "mat2vec_dev_63",
        "mat2vec_dev_73",
        "mat2vec_dev_97",
        "mat2vec_dev_121",
        "mat2vec_dev_124",
        "mat2vec_dev_154",
        "mat2vec_dev_166",
        "Processing_FinalHT_Time",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("iter1_suggestions.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    X["Processing_FinalHT_Temp"] = temp
    X["Processing_FinalHT_Time"] = time
    X = X[
        [
            "jarvis_avg_atom_mass_mult_voro_coord",
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_first_ion_en_subs_polzbl",
            "jarvis_avg_hfus_divi_voro_coord",
            "jarvis_avg_mp_add_atom_rad",
            "jarvis_avg_polzbl_divi_therm_cond",
            "jarvis_avg_polzbl_mult_atom_rad",
            "jarvis_avg_therm_cond_divi_polzbl",
            "jarvis_dev_X_subs_atom_rad",
            "jarvis_dev_atom_mass_mult_atom_rad",
            "jarvis_dev_bp_divi_polzbl",
            "jarvis_dev_mol_vol_subs_polzbl",
            "magpie_avg_CovalentRadius",
            "magpie_avg_NUnfilled",
            "oliynyk_dev_Pauling_Electronegativity",
            "oliynyk_dev_gilmor_number_of_valence_electron",
            "mat2vec_sum_7",
            "mat2vec_sum_154",
            "mat2vec_avg_7",
            "mat2vec_avg_77",
            "mat2vec_avg_84",
            "mat2vec_avg_94",
            "mat2vec_dev_35",
            "mat2vec_dev_52",
            "mat2vec_dev_63",
            "mat2vec_dev_73",
            "mat2vec_dev_97",
            "mat2vec_dev_121",
            "mat2vec_dev_124",
            "mat2vec_dev_154",
            "mat2vec_dev_166",
            "Processing_FinalHT_Time",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Ms (C)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
processed_df.to_csv("iter1_predicted_with_iter2_model_1.csv", index=False)
processed_df


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["SME_Ms"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_atom_mass_mult_voro_coord",
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_first_ion_en_subs_polzbl",
        "jarvis_avg_hfus_divi_voro_coord",
        "jarvis_avg_mp_add_atom_rad",
        "jarvis_avg_polzbl_divi_therm_cond",
        "jarvis_avg_polzbl_mult_atom_rad",
        "jarvis_avg_therm_cond_divi_polzbl",
        "jarvis_dev_X_subs_atom_rad",
        "jarvis_dev_atom_mass_mult_atom_rad",
        "jarvis_dev_bp_divi_polzbl",
        "jarvis_dev_mol_vol_subs_polzbl",
        "magpie_avg_CovalentRadius",
        "magpie_avg_NUnfilled",
        "oliynyk_dev_Pauling_Electronegativity",
        "oliynyk_dev_gilmor_number_of_valence_electron",
        "mat2vec_sum_7",
        "mat2vec_sum_154",
        "mat2vec_avg_7",
        "mat2vec_avg_77",
        "mat2vec_avg_84",
        "mat2vec_avg_94",
        "mat2vec_dev_35",
        "mat2vec_dev_52",
        "mat2vec_dev_63",
        "mat2vec_dev_73",
        "mat2vec_dev_97",
        "mat2vec_dev_121",
        "mat2vec_dev_124",
        "mat2vec_dev_154",
        "mat2vec_dev_166",
        "Processing_FinalHT_Time",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("infeasibles.csv")
df = df * 100

new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    X["Processing_FinalHT_Temp"] = temp
    X["Processing_FinalHT_Time"] = time
    X = X[
        [
            "jarvis_avg_atom_mass_mult_voro_coord",
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_first_ion_en_subs_polzbl",
            "jarvis_avg_hfus_divi_voro_coord",
            "jarvis_avg_mp_add_atom_rad",
            "jarvis_avg_polzbl_divi_therm_cond",
            "jarvis_avg_polzbl_mult_atom_rad",
            "jarvis_avg_therm_cond_divi_polzbl",
            "jarvis_dev_X_subs_atom_rad",
            "jarvis_dev_atom_mass_mult_atom_rad",
            "jarvis_dev_bp_divi_polzbl",
            "jarvis_dev_mol_vol_subs_polzbl",
            "magpie_avg_CovalentRadius",
            "magpie_avg_NUnfilled",
            "oliynyk_dev_Pauling_Electronegativity",
            "oliynyk_dev_gilmor_number_of_valence_electron",
            "mat2vec_sum_7",
            "mat2vec_sum_154",
            "mat2vec_avg_7",
            "mat2vec_avg_77",
            "mat2vec_avg_84",
            "mat2vec_avg_94",
            "mat2vec_dev_35",
            "mat2vec_dev_52",
            "mat2vec_dev_63",
            "mat2vec_dev_73",
            "mat2vec_dev_97",
            "mat2vec_dev_121",
            "mat2vec_dev_124",
            "mat2vec_dev_154",
            "mat2vec_dev_166",
            "Processing_FinalHT_Time",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Ms (C)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
processed_df.to_csv("infeasibles_1.csv", index=False)
processed_df


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["Hysteresis"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_bp_add_therm_cond",
        "jarvis_avg_bp_divi_hfus",
        "jarvis_avg_elec_aff_subs_voro_coord",
        "jarvis_avg_first_ion_en_add_voro_coord",
        "jarvis_avg_first_ion_en_mult_voro_coord",
        "jarvis_avg_hfus_add_therm_cond",
        "jarvis_avg_hfus_mult_X",
        "jarvis_avg_hfus_mult_first_ion_en",
        "jarvis_avg_mp_add_atom_mass",
        "jarvis_avg_voro_coord_divi_hfus",
        "jarvis_avg_voro_coord_subs_bp",
        "jarvis_dev_KV",
        "jarvis_dev_atom_rad_divi_bp",
        "jarvis_dev_atom_rad_subs_therm_cond",
        "jarvis_dev_elec_aff_mult_therm_cond",
        "jarvis_dev_first_ion_en_mult_mol_vol",
        "jarvis_dev_first_ion_en_mult_therm_cond",
        "jarvis_dev_first_ion_en_subs_mol_vol",
        "jarvis_dev_hfus_add_mol_vol",
        "jarvis_dev_mol_vol_subs_X",
        "jarvis_dev_polzbl_add_first_ion_en",
        "jarvis_dev_polzbl_divi_atom_mass",
        "jarvis_dev_polzbl_subs_atom_rad",
        "jarvis_dev_voro_coord",
        "jarvis_mode_atom_rad_subs_voro_coord",
        "jarvis_mode_hfus_add_first_ion_en",
        "jarvis_mode_mol_vol_add_therm_cond",
        "jarvis_mode_op_eg",
        "jarvis_mode_polzbl_subs_atom_rad",
        "jarvis_mode_therm_cond_divi_polzbl",
        "oliynyk_avg_Mulliken_EN",
        "mat2vec_avg_18",
        "mat2vec_avg_20",
        "mat2vec_avg_49",
        "mat2vec_avg_50",
        "mat2vec_avg_73",
        "mat2vec_avg_118",
        "mat2vec_avg_149",
        "mat2vec_avg_152",
        "mat2vec_avg_153",
        "mat2vec_avg_169",
        "mat2vec_dev_22",
        "mat2vec_dev_79",
        "mat2vec_dev_95",
        "mat2vec_dev_192",
        "Processing_APHT_Time",
        "SME_Test_Cycle",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("whole_space_950C_tc_included_pred_added.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    #     X["Processing_FinalHT_Temp"] = temp
    #     X["Processing_FinalHT_Time"] = time
    X["Processing_APHT_Time"] = 24.0
    X["SME_Test_Cycle"] = 2.0
    X = X[
        [
            "jarvis_avg_bp_add_therm_cond",
            "jarvis_avg_bp_divi_hfus",
            "jarvis_avg_elec_aff_subs_voro_coord",
            "jarvis_avg_first_ion_en_add_voro_coord",
            "jarvis_avg_first_ion_en_mult_voro_coord",
            "jarvis_avg_hfus_add_therm_cond",
            "jarvis_avg_hfus_mult_X",
            "jarvis_avg_hfus_mult_first_ion_en",
            "jarvis_avg_mp_add_atom_mass",
            "jarvis_avg_voro_coord_divi_hfus",
            "jarvis_avg_voro_coord_subs_bp",
            "jarvis_dev_KV",
            "jarvis_dev_atom_rad_divi_bp",
            "jarvis_dev_atom_rad_subs_therm_cond",
            "jarvis_dev_elec_aff_mult_therm_cond",
            "jarvis_dev_first_ion_en_mult_mol_vol",
            "jarvis_dev_first_ion_en_mult_therm_cond",
            "jarvis_dev_first_ion_en_subs_mol_vol",
            "jarvis_dev_hfus_add_mol_vol",
            "jarvis_dev_mol_vol_subs_X",
            "jarvis_dev_polzbl_add_first_ion_en",
            "jarvis_dev_polzbl_divi_atom_mass",
            "jarvis_dev_polzbl_subs_atom_rad",
            "jarvis_dev_voro_coord",
            "jarvis_mode_atom_rad_subs_voro_coord",
            "jarvis_mode_hfus_add_first_ion_en",
            "jarvis_mode_mol_vol_add_therm_cond",
            "jarvis_mode_op_eg",
            "jarvis_mode_polzbl_subs_atom_rad",
            "jarvis_mode_therm_cond_divi_polzbl",
            "oliynyk_avg_Mulliken_EN",
            "mat2vec_avg_18",
            "mat2vec_avg_20",
            "mat2vec_avg_49",
            "mat2vec_avg_50",
            "mat2vec_avg_73",
            "mat2vec_avg_118",
            "mat2vec_avg_149",
            "mat2vec_avg_152",
            "mat2vec_avg_153",
            "mat2vec_avg_169",
            "mat2vec_dev_22",
            "mat2vec_dev_79",
            "mat2vec_dev_95",
            "mat2vec_dev_192",
            "Processing_APHT_Time",
            "SME_Test_Cycle",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Hysteresis (C)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
processed_df


In [ ]:
pd.concat([df, processed_df[["Predicted Hysteresis (C)"]]], axis=1).to_csv(
    "whole_space_950C_tc_included_pred_added2.csv", index=False
)


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["Hysteresis"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_bp_add_therm_cond",
        "jarvis_avg_bp_divi_hfus",
        "jarvis_avg_elec_aff_subs_voro_coord",
        "jarvis_avg_first_ion_en_add_voro_coord",
        "jarvis_avg_first_ion_en_mult_voro_coord",
        "jarvis_avg_hfus_add_therm_cond",
        "jarvis_avg_hfus_mult_X",
        "jarvis_avg_hfus_mult_first_ion_en",
        "jarvis_avg_mp_add_atom_mass",
        "jarvis_avg_voro_coord_divi_hfus",
        "jarvis_avg_voro_coord_subs_bp",
        "jarvis_dev_KV",
        "jarvis_dev_atom_rad_divi_bp",
        "jarvis_dev_atom_rad_subs_therm_cond",
        "jarvis_dev_elec_aff_mult_therm_cond",
        "jarvis_dev_first_ion_en_mult_mol_vol",
        "jarvis_dev_first_ion_en_mult_therm_cond",
        "jarvis_dev_first_ion_en_subs_mol_vol",
        "jarvis_dev_hfus_add_mol_vol",
        "jarvis_dev_mol_vol_subs_X",
        "jarvis_dev_polzbl_add_first_ion_en",
        "jarvis_dev_polzbl_divi_atom_mass",
        "jarvis_dev_polzbl_subs_atom_rad",
        "jarvis_dev_voro_coord",
        "jarvis_mode_atom_rad_subs_voro_coord",
        "jarvis_mode_hfus_add_first_ion_en",
        "jarvis_mode_mol_vol_add_therm_cond",
        "jarvis_mode_op_eg",
        "jarvis_mode_polzbl_subs_atom_rad",
        "jarvis_mode_therm_cond_divi_polzbl",
        "oliynyk_avg_Mulliken_EN",
        "mat2vec_avg_18",
        "mat2vec_avg_20",
        "mat2vec_avg_49",
        "mat2vec_avg_50",
        "mat2vec_avg_73",
        "mat2vec_avg_118",
        "mat2vec_avg_149",
        "mat2vec_avg_152",
        "mat2vec_avg_153",
        "mat2vec_avg_169",
        "mat2vec_dev_22",
        "mat2vec_dev_79",
        "mat2vec_dev_95",
        "mat2vec_dev_192",
        "Processing_APHT_Time",
        "SME_Test_Cycle",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("iter1_predicted_with_iter2_model_1.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    #     X["Processing_FinalHT_Temp"] = temp
    #     X["Processing_FinalHT_Time"] = time
    X["Processing_APHT_Time"] = 24.0
    X["SME_Test_Cycle"] = 2.0
    X = X[
        [
            "jarvis_avg_bp_add_therm_cond",
            "jarvis_avg_bp_divi_hfus",
            "jarvis_avg_elec_aff_subs_voro_coord",
            "jarvis_avg_first_ion_en_add_voro_coord",
            "jarvis_avg_first_ion_en_mult_voro_coord",
            "jarvis_avg_hfus_add_therm_cond",
            "jarvis_avg_hfus_mult_X",
            "jarvis_avg_hfus_mult_first_ion_en",
            "jarvis_avg_mp_add_atom_mass",
            "jarvis_avg_voro_coord_divi_hfus",
            "jarvis_avg_voro_coord_subs_bp",
            "jarvis_dev_KV",
            "jarvis_dev_atom_rad_divi_bp",
            "jarvis_dev_atom_rad_subs_therm_cond",
            "jarvis_dev_elec_aff_mult_therm_cond",
            "jarvis_dev_first_ion_en_mult_mol_vol",
            "jarvis_dev_first_ion_en_mult_therm_cond",
            "jarvis_dev_first_ion_en_subs_mol_vol",
            "jarvis_dev_hfus_add_mol_vol",
            "jarvis_dev_mol_vol_subs_X",
            "jarvis_dev_polzbl_add_first_ion_en",
            "jarvis_dev_polzbl_divi_atom_mass",
            "jarvis_dev_polzbl_subs_atom_rad",
            "jarvis_dev_voro_coord",
            "jarvis_mode_atom_rad_subs_voro_coord",
            "jarvis_mode_hfus_add_first_ion_en",
            "jarvis_mode_mol_vol_add_therm_cond",
            "jarvis_mode_op_eg",
            "jarvis_mode_polzbl_subs_atom_rad",
            "jarvis_mode_therm_cond_divi_polzbl",
            "oliynyk_avg_Mulliken_EN",
            "mat2vec_avg_18",
            "mat2vec_avg_20",
            "mat2vec_avg_49",
            "mat2vec_avg_50",
            "mat2vec_avg_73",
            "mat2vec_avg_118",
            "mat2vec_avg_149",
            "mat2vec_avg_152",
            "mat2vec_avg_153",
            "mat2vec_avg_169",
            "mat2vec_dev_22",
            "mat2vec_dev_79",
            "mat2vec_dev_95",
            "mat2vec_dev_192",
            "Processing_APHT_Time",
            "SME_Test_Cycle",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Hysteresis (C)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
pd.concat([df, processed_df[["Predicted Hysteresis (C)"]]], axis=1).to_csv(
    "iter1_predicted_with_iter2_model_2.csv", index=False
)


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["Hysteresis"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_bp_add_therm_cond",
        "jarvis_avg_bp_divi_hfus",
        "jarvis_avg_elec_aff_subs_voro_coord",
        "jarvis_avg_first_ion_en_add_voro_coord",
        "jarvis_avg_first_ion_en_mult_voro_coord",
        "jarvis_avg_hfus_add_therm_cond",
        "jarvis_avg_hfus_mult_X",
        "jarvis_avg_hfus_mult_first_ion_en",
        "jarvis_avg_mp_add_atom_mass",
        "jarvis_avg_voro_coord_divi_hfus",
        "jarvis_avg_voro_coord_subs_bp",
        "jarvis_dev_KV",
        "jarvis_dev_atom_rad_divi_bp",
        "jarvis_dev_atom_rad_subs_therm_cond",
        "jarvis_dev_elec_aff_mult_therm_cond",
        "jarvis_dev_first_ion_en_mult_mol_vol",
        "jarvis_dev_first_ion_en_mult_therm_cond",
        "jarvis_dev_first_ion_en_subs_mol_vol",
        "jarvis_dev_hfus_add_mol_vol",
        "jarvis_dev_mol_vol_subs_X",
        "jarvis_dev_polzbl_add_first_ion_en",
        "jarvis_dev_polzbl_divi_atom_mass",
        "jarvis_dev_polzbl_subs_atom_rad",
        "jarvis_dev_voro_coord",
        "jarvis_mode_atom_rad_subs_voro_coord",
        "jarvis_mode_hfus_add_first_ion_en",
        "jarvis_mode_mol_vol_add_therm_cond",
        "jarvis_mode_op_eg",
        "jarvis_mode_polzbl_subs_atom_rad",
        "jarvis_mode_therm_cond_divi_polzbl",
        "oliynyk_avg_Mulliken_EN",
        "mat2vec_avg_18",
        "mat2vec_avg_20",
        "mat2vec_avg_49",
        "mat2vec_avg_50",
        "mat2vec_avg_73",
        "mat2vec_avg_118",
        "mat2vec_avg_149",
        "mat2vec_avg_152",
        "mat2vec_avg_153",
        "mat2vec_avg_169",
        "mat2vec_dev_22",
        "mat2vec_dev_79",
        "mat2vec_dev_95",
        "mat2vec_dev_192",
        "Processing_APHT_Time",
        "SME_Test_Cycle",
    ]
]
X_ms
params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("infeasibles_1.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    #     X["Processing_FinalHT_Temp"] = temp
    #     X["Processing_FinalHT_Time"] = time
    X["Processing_APHT_Time"] = 24.0
    X["SME_Test_Cycle"] = 2.0
    X = X[
        [
            "jarvis_avg_bp_add_therm_cond",
            "jarvis_avg_bp_divi_hfus",
            "jarvis_avg_elec_aff_subs_voro_coord",
            "jarvis_avg_first_ion_en_add_voro_coord",
            "jarvis_avg_first_ion_en_mult_voro_coord",
            "jarvis_avg_hfus_add_therm_cond",
            "jarvis_avg_hfus_mult_X",
            "jarvis_avg_hfus_mult_first_ion_en",
            "jarvis_avg_mp_add_atom_mass",
            "jarvis_avg_voro_coord_divi_hfus",
            "jarvis_avg_voro_coord_subs_bp",
            "jarvis_dev_KV",
            "jarvis_dev_atom_rad_divi_bp",
            "jarvis_dev_atom_rad_subs_therm_cond",
            "jarvis_dev_elec_aff_mult_therm_cond",
            "jarvis_dev_first_ion_en_mult_mol_vol",
            "jarvis_dev_first_ion_en_mult_therm_cond",
            "jarvis_dev_first_ion_en_subs_mol_vol",
            "jarvis_dev_hfus_add_mol_vol",
            "jarvis_dev_mol_vol_subs_X",
            "jarvis_dev_polzbl_add_first_ion_en",
            "jarvis_dev_polzbl_divi_atom_mass",
            "jarvis_dev_polzbl_subs_atom_rad",
            "jarvis_dev_voro_coord",
            "jarvis_mode_atom_rad_subs_voro_coord",
            "jarvis_mode_hfus_add_first_ion_en",
            "jarvis_mode_mol_vol_add_therm_cond",
            "jarvis_mode_op_eg",
            "jarvis_mode_polzbl_subs_atom_rad",
            "jarvis_mode_therm_cond_divi_polzbl",
            "oliynyk_avg_Mulliken_EN",
            "mat2vec_avg_18",
            "mat2vec_avg_20",
            "mat2vec_avg_49",
            "mat2vec_avg_50",
            "mat2vec_avg_73",
            "mat2vec_avg_118",
            "mat2vec_avg_149",
            "mat2vec_avg_152",
            "mat2vec_avg_153",
            "mat2vec_avg_169",
            "mat2vec_dev_22",
            "mat2vec_dev_79",
            "mat2vec_dev_95",
            "mat2vec_dev_192",
            "Processing_APHT_Time",
            "SME_Test_Cycle",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Hysteresis (C)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
pd.concat([df, processed_df[["Predicted Hysteresis (C)"]]], axis=1).to_csv(
    "infeasibles_2.csv", index=False
)


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]
df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["Enthalpy"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_mol_vol_divi_therm_cond",
        "jarvis_avg_voro_coord_divi_bp",
        "jarvis_dev_voro_coord_subs_mol_vol",
        "magpie_avg_NUnfilled",
        "mat2vec_avg_133",
    ]
]

params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("whole_space_950C_tc_included_pred_added2.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    #     X["Processing_FinalHT_Temp"] = temp
    #     X["Processing_FinalHT_Time"] = time
    #     X["Processing_APHT_Time"] = 24.0
    #     X["SME_Test_Cycle"] = 2.0
    X = X[
        [
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_mol_vol_divi_therm_cond",
            "jarvis_avg_voro_coord_divi_bp",
            "jarvis_dev_voro_coord_subs_mol_vol",
            "magpie_avg_NUnfilled",
            "mat2vec_avg_133",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Enthalpy (J/g)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
processed_df


In [ ]:
pd.concat([df, processed_df[["Predicted Enthalpy (J/g)"]]], axis=1).to_csv(
    "whole_space_950C_tc_included_pred_added3.csv", index=False
)


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]
df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["Enthalpy"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_mol_vol_divi_therm_cond",
        "jarvis_avg_voro_coord_divi_bp",
        "jarvis_dev_voro_coord_subs_mol_vol",
        "magpie_avg_NUnfilled",
        "mat2vec_avg_133",
    ]
]

params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("iter1_predicted_with_iter2_model_2.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    #     X["Processing_FinalHT_Temp"] = temp
    #     X["Processing_FinalHT_Time"] = time
    #     X["Processing_APHT_Time"] = 24.0
    #     X["SME_Test_Cycle"] = 2.0
    X = X[
        [
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_mol_vol_divi_therm_cond",
            "jarvis_avg_voro_coord_divi_bp",
            "jarvis_dev_voro_coord_subs_mol_vol",
            "magpie_avg_NUnfilled",
            "mat2vec_avg_133",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Enthalpy (J/g)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
pd.concat([df, processed_df[["Predicted Enthalpy (J/g)"]]], axis=1).to_csv(
    "iter1_predicted_with_iter2_model_3.csv", index=False
)


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    LITERATURE_SPREADSHEET_URL,
    "SMA_Database",
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
    & (df["Te(at%)"] == 0)
]

df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df.columns = [x.replace("(at%)", "") for x in df.columns]

df_lit = df[
    [
        "Ag",
        "Al",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Co",
        "Cr",
        "Cu",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "Hf",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Ni",
        "Pb",
        "Pd",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Ti",
        "Tl",
        "V",
        "W",
        "Y",
        "Zr",
        #         "Processing_BPHT_Temp",
        #         "Processing_BPHT_Time",
        #         "Processing_Rolling_Temp",
        #         "Processing_HR_Red",
        #         "Processing_CR_Red",
        #         "Processing_Extrusion_Temp",
        #         "Processing_Extrusion_Area_Reduction(%)",
        #         "Processing_ECAE_Temp",
        #         "Processing_ECAE_Route",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        #         "SME_Test_Applied_Stress(MPa)",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]
df_lit.loc[:, "Total"] = df_lit[df_lit.iloc[:, :40].columns.to_list()].sum(axis=1)
df_lit[(df_lit.Total >= 100.5) & (df_lit.Total <= 99.5)].to_csv("drops_output.csv")
df_lit = df_lit.drop("Total", axis=1)
df_lit["Hysteresis"] = df_lit["SME_Af"] - df_lit["SME_Ms"]
df_lit.drop_duplicates(inplace=True)
df_lit.reset_index(inplace=True, drop=True)


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")

df = df[
    (df["SME_Ms"] < df["SME_Af"])
    & (df["SME_As"] < df["SME_Af"])
    & (df["SME_Mf"] < df["SME_Ms"])
    & (df["SME_Mf"] < df["SME_As"])
    & (df["SME_Ms"] > -273.15)
    & (df["SME_Af"] > -273.15)
    & (df["SME_As"] > -273.15)
    & (df["SME_Mf"] > -273.15)
]
df["Enthalpy"] = (df["SME_dH(MA)(J/g)"] + df["SME_dH(AM)(J/g)"]) / 2
df = df[df["Enthalpy"] > 0]

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "Processing_APHT_Temp",
        "Processing_APHT_Time",
        "Processing_FinalHT_Temp",
        "Processing_FinalHT_Time",
        "SME_Test_Cycle",
        "SME_Ms",
        "SME_Mf",
        "SME_Af",
        "SME_As",
        "Enthalpy",
    ]
]

df_iter1.loc[:, "Total"] = df_iter1[df_iter1.iloc[:, :40].columns.to_list()].sum(axis=1)
df_iter1[(df_iter1.Total >= 100.5) & (df_iter1.Total <= 99.5)].to_csv(
    "drops_output2.csv"
)
df_iter1 = df_iter1.drop("Total", axis=1)
df_iter1["Hysteresis"] = df_iter1["SME_Af"] - df_iter1["SME_Ms"]
df_iter1.drop_duplicates(inplace=True)
df_iter1.reset_index(inplace=True, drop=True)

df_iter1 = df_iter1.astype("float64")
df_lit = df_lit.astype("float64")


# Merge the dataframes
df_merged = pd.merge(df_lit, df_iter1, how="outer").fillna(0)

# Display the resulting dataframe
y = df_merged[["Enthalpy"]]
X_elems = df_merged.iloc[:, :40]
analyzer = FeatureGenerator(X_elems)
comp_df = analyzer.generate_composition_formula()
X_features = analyzer.generate_features_all()
X = pd.concat([X_features, df_merged.iloc[:, 40:45]], axis=1)
X_ms = X[
    [
        "jarvis_avg_first_ion_en_add_elec_aff",
        "jarvis_avg_mol_vol_divi_therm_cond",
        "jarvis_avg_voro_coord_divi_bp",
        "jarvis_dev_voro_coord_subs_mol_vol",
        "magpie_avg_NUnfilled",
        "mat2vec_avg_133",
    ]
]

params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(**params)
model.fit(X_ms, y)


# Use the model fitted in the preceding training section.
temp = 950
time = 24

# Read the entire CSV file at once
df = pd.read_csv("infeasibles_2.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate descriptors and return the configured predictions for one batch."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    X = analyzer.generate_features_all()

    #     X["Processing_FinalHT_Temp"] = temp
    #     X["Processing_FinalHT_Time"] = time
    #     X["Processing_APHT_Time"] = 24.0
    #     X["SME_Test_Cycle"] = 2.0
    X = X[
        [
            "jarvis_avg_first_ion_en_add_elec_aff",
            "jarvis_avg_mol_vol_divi_therm_cond",
            "jarvis_avg_voro_coord_divi_bp",
            "jarvis_dev_voro_coord_subs_mol_vol",
            "magpie_avg_NUnfilled",
            "mat2vec_avg_133",
        ]
    ]

    # Process the DataFrame with your FeatureGenerator and model prediction
    pred = model.predict(X)

    # Combine the predictions with the DataFrame
    df = pd.concat(
        [
            df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]],
            pd.DataFrame(
                pred,
                columns=[
                    "Predicted Enthalpy (J/g)",
                ],
            ),
        ],
        axis=1,
    )
    return df


# Process the DataFrame
processed_df = process_dataframe(new_df)
pd.concat([df, processed_df[["Predicted Enthalpy (J/g)"]]], axis=1).to_csv(
    "infeasibles_3.csv", index=False
)


## Transformation-strain analysis

Generate lattice-compatibility features, combine the literature and experimental
strain datasets, and fit the strain model before applying it to candidate tables.

In [ ]:
import sys
from importlib.metadata import version

import ast
import pandas as pd
import numpy as np


from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("Python version: " + sys.version)
print("NumPy version: {}".format(version("numpy")))
print("pandas version: {}".format(version("pandas")))
print("CBFV version: {}".format(version("CBFV")))


In [ ]:
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


In [ ]:
# Prepare the first experimental iteration for transformation-strain modeling.
import gspread
import numpy as np
import pandas as pd
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_1_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")


df = df.dropna(subset=["a0 (A)"])

df_iter1 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "a0 (A)",
        "a (A)",
        "b (A)",
        "c (A)",
        "beta",
    ]
]

df_iter1.reset_index(inplace=True, drop=True)

directions = [
    [1, 1, 1],
    [0, 1, 1],
    [1, 1, 2],
    [1, 4, 8],
    [2, -2, -1],
    [8, -8, -1],
    [4, -4, 1],
    [3, -3, 2],
    [8, -8, 3],
    [5, -5, 4],
    [5, -5, 7],
    [2, -2, 5],
]


results = []

for index, row in df_iter1.iterrows():
    # Initialize the calculator
    calculator = TransformationStrainCalculator()

    for deformation_direction in directions:
        # Setup calculator with data from the row
        calculator.set_lattice_constants_and_beta(
            row["a0 (A)"], row["a (A)"], row["b (A)"], row["c (A)"], row["beta"]
        )
        calculator.set_custom_directions(deformation_direction)

        # Calculate max strain and other information
        max_strains_info = calculator.calc_max_strain_and_info()

        # Append row for tension
        tension_row = row.copy()
        tension_row["deformation_direction"] = str(deformation_direction)
        tension_row["deformation_type"] = "tension"
        tension_row["LDT_transformation_strain"] = max_strains_info[0][0]
        results.append(tension_row)

        # Append row for compression
        compression_row = row.copy()
        compression_row["deformation_direction"] = str(deformation_direction)
        compression_row["deformation_type"] = "compression"
        compression_row["LDT_transformation_strain"] = max_strains_info[0][1]
        results.append(compression_row)

#     if index == 1:  # Ensure this is your actual stopping condition
#         break

# Convert the results list to a DataFrame
new_df = pd.DataFrame(results)

# Display the new DataFrame
new_df.reset_index(drop=True, inplace=True)
new_df.to_csv("iter1_transformation_strain.csv", index=False)
new_df


In [ ]:
# Prepare the second experimental iteration using the same strain columns.
pd.options.mode.chained_assignment = None  # default='warn'


# Load the selected worksheet; numeric conversion is specific to this analysis.


df = get_dataframe(
    ITERATION_2_SPREADSHEET_URL,
    "Sheet1",
)
df = df.apply(pd.to_numeric, errors="ignore")


df = df.dropna(subset=["a0 (A)"])

df_iter2 = df[
    [
        "Ni",
        "Ti",
        "Cu",
        "Hf",
        "Zr",
        "Pd",
        "Co",
        "a0 (A)",
        "a (A)",
        "b (A)",
        "c (A)",
        "beta",
    ]
]

df_iter2.reset_index(inplace=True, drop=True)

directions = [
    [1, 1, 1],
    [0, 1, 1],
    [1, 1, 2],
    [1, 4, 8],
    [2, -2, -1],
    [8, -8, -1],
    [4, -4, 1],
    [3, -3, 2],
    [8, -8, 3],
    [5, -5, 4],
    [5, -5, 7],
    [2, -2, 5],
]


results = []

for index, row in df_iter2.iterrows():
    # Initialize the calculator
    calculator = TransformationStrainCalculator()

    for deformation_direction in directions:
        # Setup calculator with data from the row
        calculator.set_lattice_constants_and_beta(
            row["a0 (A)"], row["a (A)"], row["b (A)"], row["c (A)"], row["beta"]
        )
        calculator.set_custom_directions(deformation_direction)

        # Calculate max strain and other information
        max_strains_info = calculator.calc_max_strain_and_info()

        # Append row for tension
        tension_row = row.copy()
        tension_row["deformation_direction"] = str(deformation_direction)
        tension_row["deformation_type"] = "tension"
        tension_row["LDT_transformation_strain"] = max_strains_info[0][0]
        results.append(tension_row)

        # Append row for compression
        compression_row = row.copy()
        compression_row["deformation_direction"] = str(deformation_direction)
        compression_row["deformation_type"] = "compression"
        compression_row["LDT_transformation_strain"] = max_strains_info[0][1]
        results.append(compression_row)

#     if index == 1:  # Ensure this is your actual stopping condition
#         break

# Convert the results list to a DataFrame
new_df = pd.DataFrame(results)

# Display the new DataFrame
new_df.reset_index(drop=True, inplace=True)
new_df.iloc[:, :7] = new_df.iloc[:, :7] * 100
new_df.to_csv("iter2_transformation_strain.csv", index=False)
new_df


In [ ]:
df = pd.read_csv("iter1_transformation_strain.csv")
df.iloc[:, :7]


In [ ]:
df = pd.read_csv("iter2_transformation_strain.csv")
df.iloc[:, :7] = df.iloc[:, :7] * 100
df.iloc[:, :7]


In [ ]:
# Fit strain using lambda2, deformation type, and the deformation-direction embedding.
import sys
from importlib.metadata import version

import ast
import pandas as pd
import numpy as np


from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import gspread
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime


print("Python version: " + sys.version)
print("NumPy version: {}".format(version("numpy")))
print("pandas version: {}".format(version("pandas")))
print("CBFV version: {}".format(version("CBFV")))

df = pd.read_csv("iter1_transformation_strain.csv")

feature_generator = FeatureGenerator(df.iloc[:, :7])

comp_df = feature_generator.generate_composition_formula()
features_df = feature_generator.generate_features()

lambda2_model = Lambda2Model(features_df)

prediction_df = lambda2_model.predict_transformation_and_lambda2()
df_iter1 = pd.concat(
    [
        prediction_df[["Predicted_Lambda2"]],
        df[["deformation_direction", "deformation_type", "LDT_transformation_strain"]],
    ],
    axis=1,
)
df_iter1["deformation_direction"] = df_iter1["deformation_direction"].apply(
    ast.literal_eval
)


df = pd.read_csv("iter2_transformation_strain.csv")
df.iloc[:, :7] = df.iloc[:, :7] * 100
feature_generator = FeatureGenerator(df.iloc[:, :7])
comp_df = feature_generator.generate_composition_formula()
features_df = feature_generator.generate_features()

lambda2_model = Lambda2Model(features_df)

prediction_df = lambda2_model.predict_transformation_and_lambda2()
df_iter2 = pd.concat(
    [
        prediction_df[["Predicted_Lambda2"]],
        df[["deformation_direction", "deformation_type", "LDT_transformation_strain"]],
    ],
    axis=1,
)
df_iter2["deformation_direction"] = df_iter2["deformation_direction"].apply(
    ast.literal_eval
)


# Read data
df_lit = pd.read_csv("data_for_ml_ts.csv")
df_lit["deformation_direction"] = df_lit["deformation_direction"].apply(
    ast.literal_eval
)

# Select columns
df_lit = df_lit[
    [
        "Predicted_Lambda2",
        "deformation_direction",
        "deformation_type",
        "LDT_transformation_strain",
    ]
].reset_index(drop=True)

# Split data into features and target


df_merged = pd.concat([df_lit, df_iter1, df_iter2], axis=0)
df_merged.reset_index(drop=True, inplace=True)
X = df_merged[["Predicted_Lambda2", "deformation_direction", "deformation_type"]]
y = df_merged[["LDT_transformation_strain"]]


params = {"loss_function": "MAE", "eval_metric": "MAE", "silent": True}
model = cb.CatBoostRegressor(
    **params,
    cat_features=["deformation_type"],
    embedding_features=["deformation_direction"],
)
model.fit(X, y)


In [ ]:
df = pd.read_csv("whole_space_950C_tc_included_pred_added3.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]
new_df
feature_generator = FeatureGenerator(new_df)
comp_df = feature_generator.generate_composition_formula()
features_df = feature_generator.generate_features()

lambda2_model = Lambda2Model(features_df)
prediction_df = lambda2_model.predict_transformation_and_lambda2()
df_q = prediction_df[["Predicted_Lambda2"]].copy()
df_q["deformation_direction"] = "[0, 1, 1]"
df_q["deformation_direction"] = df_q["deformation_direction"].apply(ast.literal_eval)
df_q["deformation_type"] = "tension"
pred = model.predict(df_q)
pred_series = pd.Series(pred, name="predicted_transformation_strain")

# Concatenate df and pred_series
result = pd.concat([df, pred_series], axis=1)

# Save the concatenated DataFrame to a CSV file
result.to_csv("whole_space_950C_tc_included_pred_added4.csv", index=False)


In [ ]:
df = pd.read_csv("iter1_predicted_with_iter2_model_3.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]
new_df
feature_generator = FeatureGenerator(new_df)
comp_df = feature_generator.generate_composition_formula()
features_df = feature_generator.generate_features()

lambda2_model = Lambda2Model(features_df)
prediction_df = lambda2_model.predict_transformation_and_lambda2()
df_q = prediction_df[["Predicted_Lambda2"]].copy()
df_q["deformation_direction"] = "[0, 1, 1]"
df_q["deformation_direction"] = df_q["deformation_direction"].apply(ast.literal_eval)
df_q["deformation_type"] = "tension"
pred = model.predict(df_q)
pred_series = pd.Series(pred, name="predicted_transformation_strain")

# Concatenate df and pred_series
result = pd.concat([df, pred_series], axis=1)

# Save the concatenated DataFrame to a CSV file
result.to_csv("iter1_predicted_with_iter2_model_4.csv", index=False)


In [ ]:
df = pd.read_csv("infeasibles_3.csv")
new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]
new_df
feature_generator = FeatureGenerator(new_df)
comp_df = feature_generator.generate_composition_formula()
features_df = feature_generator.generate_features()

lambda2_model = Lambda2Model(features_df)
prediction_df = lambda2_model.predict_transformation_and_lambda2()
df_q = prediction_df[["Predicted_Lambda2"]].copy()
df_q["deformation_direction"] = "[0, 1, 1]"
df_q["deformation_direction"] = df_q["deformation_direction"].apply(ast.literal_eval)
df_q["deformation_type"] = "tension"
pred = model.predict(df_q)
pred_series = pd.Series(pred, name="predicted_transformation_strain")

# Concatenate df and pred_series
result = pd.concat([df, pred_series], axis=1)

# Save the concatenated DataFrame to a CSV file
result.to_csv("infeasibles_4.csv", index=False)


## Export candidate features

Combine the descriptor names selected for each property, generate features for
the candidate compositions, and export the final table for downstream analysis.

In [ ]:
# Collect the descriptor names used across the property models.
el_list = ["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]
ms_features = [
    "jarvis_avg_atom_mass_mult_voro_coord",
    "jarvis_avg_first_ion_en_add_elec_aff",
    "jarvis_avg_first_ion_en_subs_polzbl",
    "jarvis_avg_hfus_divi_voro_coord",
    "jarvis_avg_mp_add_atom_rad",
    "jarvis_avg_polzbl_divi_therm_cond",
    "jarvis_avg_polzbl_mult_atom_rad",
    "jarvis_avg_therm_cond_divi_polzbl",
    "jarvis_dev_X_subs_atom_rad",
    "jarvis_dev_atom_mass_mult_atom_rad",
    "jarvis_dev_bp_divi_polzbl",
    "jarvis_dev_mol_vol_subs_polzbl",
    "magpie_avg_CovalentRadius",
    "magpie_avg_NUnfilled",
    "oliynyk_dev_Pauling_Electronegativity",
    "oliynyk_dev_gilmor_number_of_valence_electron",
    "mat2vec_sum_7",
    "mat2vec_sum_154",
    "mat2vec_avg_7",
    "mat2vec_avg_77",
    "mat2vec_avg_84",
    "mat2vec_avg_94",
    "mat2vec_dev_35",
    "mat2vec_dev_52",
    "mat2vec_dev_63",
    "mat2vec_dev_73",
    "mat2vec_dev_97",
    "mat2vec_dev_121",
    "mat2vec_dev_124",
    "mat2vec_dev_154",
    "mat2vec_dev_166",
]
hst_features = [
    "jarvis_avg_bp_add_therm_cond",
    "jarvis_avg_bp_divi_hfus",
    "jarvis_avg_elec_aff_subs_voro_coord",
    "jarvis_avg_first_ion_en_add_voro_coord",
    "jarvis_avg_first_ion_en_mult_voro_coord",
    "jarvis_avg_hfus_add_therm_cond",
    "jarvis_avg_hfus_mult_X",
    "jarvis_avg_hfus_mult_first_ion_en",
    "jarvis_avg_mp_add_atom_mass",
    "jarvis_avg_voro_coord_divi_hfus",
    "jarvis_avg_voro_coord_subs_bp",
    "jarvis_dev_KV",
    "jarvis_dev_atom_rad_divi_bp",
    "jarvis_dev_atom_rad_subs_therm_cond",
    "jarvis_dev_elec_aff_mult_therm_cond",
    "jarvis_dev_first_ion_en_mult_mol_vol",
    "jarvis_dev_first_ion_en_mult_therm_cond",
    "jarvis_dev_first_ion_en_subs_mol_vol",
    "jarvis_dev_hfus_add_mol_vol",
    "jarvis_dev_mol_vol_subs_X",
    "jarvis_dev_polzbl_add_first_ion_en",
    "jarvis_dev_polzbl_divi_atom_mass",
    "jarvis_dev_polzbl_subs_atom_rad",
    "jarvis_dev_voro_coord",
    "jarvis_mode_atom_rad_subs_voro_coord",
    "jarvis_mode_hfus_add_first_ion_en",
    "jarvis_mode_mol_vol_add_therm_cond",
    "jarvis_mode_op_eg",
    "jarvis_mode_polzbl_subs_atom_rad",
    "jarvis_mode_therm_cond_divi_polzbl",
    "oliynyk_avg_Mulliken_EN",
    "mat2vec_avg_18",
    "mat2vec_avg_20",
    "mat2vec_avg_49",
    "mat2vec_avg_50",
    "mat2vec_avg_73",
    "mat2vec_avg_118",
    "mat2vec_avg_149",
    "mat2vec_avg_152",
    "mat2vec_avg_153",
    "mat2vec_avg_169",
    "mat2vec_dev_22",
    "mat2vec_dev_79",
    "mat2vec_dev_95",
    "mat2vec_dev_192",
]

ent_features = [
    "jarvis_avg_first_ion_en_add_elec_aff",
    "jarvis_avg_mol_vol_divi_therm_cond",
    "jarvis_avg_voro_coord_divi_bp",
    "jarvis_dev_voro_coord_subs_mol_vol",
    "magpie_avg_NUnfilled",
    "mat2vec_avg_133",
]

ts_features = ["Predicted_Lambda2"]

combined_unique_list = el_list + list(
    set(ms_features + hst_features + ent_features + ts_features)
)
combined_unique_list


In [ ]:
# Convert candidate fractions to atomic percent and export the combined feature table.
import sys
from importlib.metadata import version

import ast
import pandas as pd
import numpy as np


from helper import (
    TransformationStrainCalculator,
    DataHandler,
    FeatureGenerator,
    deformation_matrices,
    LambdaCalculator,
    Lambda2Model,
)


from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import gspread
from oauth2client.service_account import ServiceAccountCredentials
from helper import FeatureGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import catboost as cb
import shap
from datetime import datetime

df = pd.read_csv("all_candidates.csv")
df = df * 100

new_df = df[["Ni", "Ti", "Cu", "Hf", "Zr", "Pd", "Co"]]


# Process the entire DataFrame
def process_dataframe(df):
    # Initialize or update the necessary columns
    """Generate the combined model descriptors for candidate compositions."""
    for element in [
        "Ag",
        "Au",
        "B",
        "Bi",
        "Ce",
        "Cr",
        "Dy",
        "Er",
        "Fe",
        "Ga",
        "Gd",
        "In",
        "La",
        "Mn",
        "Mo",
        "Nb",
        "Nd",
        "Pb",
        "Pr",
        "Pt",
        "Re",
        "Rh",
        "Sb",
        "Sc",
        "Si",
        "Sn",
        "Ta",
        "Te",
        "Tl",
        "W",
        "Y",
        "Al",
        "V",
    ]:
        if element not in df.columns:
            df[element] = 0.0

    # Set processing conditions
    #     df["Processing_APHT_Temp"] = 1100.0
    #     df["Processing_APHT_Time"] = 24.0
    #     df["SME_Test_Applied_Stress(MPa)"] = 0.0
    #     df["SME_Test_Cycle"] = 2.0
    analyzer = FeatureGenerator(df)
    analyzer.generate_composition_formula()
    features_df = analyzer.generate_features_all()
    lambda2_model = Lambda2Model(features_df)
    X = lambda2_model.predict_transformation_and_lambda2()
    X = X[combined_unique_list]
    return X


processed_df = process_dataframe(new_df)
processed_df.to_csv("all_candidates_features_added.csv", index=False)
processed_df
